# Indices Signal Exploration

Interactive notebook for exploring momentum signals on equity indices.

**Goals:**
1. Load and explore indices data
2. Compute momentum factors (6M returns)
3. Calculate cross-sectional ranks
4. Test different signal parameters
5. Visualize long/short signals over time

In [ ]:
# Imports - PYTHON PHASE
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from IPython.display import display, HTML

# Plotly imports
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

# Set display options
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', 20)
pd.set_option('display.width', None)

# Plotting style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

print("✅ Imports loaded")

ModuleNotFoundError: No module named 'pandas'

## 1. Load Indices Data

The `lagging_indecies.py` example uses `parquet://equities/indicies.parquet`

ssa


In [ ]:
# Load indices data - EXPANDED DATA LOADING
data_path = Path("../equities/indicies.parquet")

# Check if file exists
if not data_path.exists():
    print(f"⚠️ File not found at {data_path.resolve()}")
    print(f"Current working directory: {Path.cwd()}")
    print("Available files in equities/:")
    equities_path = Path("../equities")
    if equities_path.exists():
        for f in equities_path.iterdir():
            print(f"  - {f.name}")
else:
    # Load the data
    df_raw = pd.read_parquet(data_path)
    
    print(f"✅ Data loaded successfully")
    print(f"Shape: {df_raw.shape}")
    print(f"Columns: {df_raw.columns.tolist()}")
    print(f"Data types:\n{df_raw.dtypes}")
    
    # Check if there's a datetime column that needs to be set as index
    datetime_cols = df_raw.select_dtypes(include=['datetime64']).columns
    if len(datetime_cols) > 0:
        print(f"\n✅ Found datetime column: {datetime_cols[0]}")
        df_raw = df_raw.set_index(datetime_cols[0])
        print(f"Index set to: {df_raw.index.name}")
    
    # Display first few rows
    print(f"\n📊 First few rows:")
    display(df_raw.head(10))

NameError: name 'Path' is not defined

In [ ]:
# Explore data structure - COMPREHENSIVE DATA ANALYSIS
print("="*60)
print("DATA EXPLORATION")
print("="*60)

print("\n📅 DATE RANGE:")
print(f"  From: {df_raw.index.min()}")
print(f"  To: {df_raw.index.max()}")
print(f"  Total days: {len(df_raw)}")
print(f"  Frequency: ~{len(df_raw) / ((df_raw.index.max() - df_raw.index.min()).days / 365):.1f} obs per year")

print("\n📊 INSTRUMENTS (COLUMNS):")
print(f"  {df_raw.columns.tolist()}")
print(f"  Total: {len(df_raw.columns)}")

print("\n🔍 MISSING DATA:")
missing = df_raw.isnull().sum()
if missing.sum() > 0:
    print(f"  Found missing values:")
    print(missing[missing > 0])
else:
    print(f"  ✅ No missing data")

print("\n📈 PRICE STATISTICS:")
# Select only numeric columns for describe
numeric_df = df_raw.select_dtypes(include=[np.number])
if len(numeric_df.columns) > 0:
    display(numeric_df.describe())
else:
    print("  No numeric columns found")

print("\n🔗 CORRELATION:")
if len(numeric_df.columns) > 1:
    corr = numeric_df.corr()
    display(corr)
else:
    print("  Need at least 2 numeric columns for correlation")

DATA EXPLORATION

📅 DATE RANGE:
  From: 1983-12-30 00:00:00
  To: 2025-12-12 00:00:00
  Total days: 60370
  Frequency: ~1438.0 obs per year

📊 INSTRUMENTS (COLUMNS):
  ['ticker', 'open', 'high', 'low', 'close', 'volume']
  Total: 6

🔍 MISSING DATA:
  ✅ No missing data

📈 PRICE STATISTICS:


,open,high,low,close,volume
count,60370.000000,60370.000000,60370.000000,60370.000000,6.037000e+04
mean,7970.022375,8018.557988,7916.093603,7969.020073,4.116153e+08
std,8465.870692,8499.333951,8426.692137,8464.027765,4.815778e+08
min,147.820000,149.280000,147.260000,147.820000,0.000000e+00
25%,2381.017500,2394.562500,2363.385000,2378.775000,4.496163e+07
50%,5379.620000,5416.515000,5344.605000,5382.330000,2.014988e+08
75%,9471.630000,9540.130000,9400.397500,9470.042500,6.428947e+08
max,50109.000000,50109.000000,50109.000000,50109.000000,4.463321e+09



🔗 CORRELATION:


,open,high,low,close,volume
open,1.000000,0.999962,0.999945,0.999919,0.005841
high,0.999962,1.000000,0.999914,0.999950,0.006855
low,0.999945,0.999914,1.000000,0.999959,0.004520
close,0.999919,0.999950,0.999959,1.000000,0.005644
volume,0.005841,0.006855,0.004520,0.005644,1.000000


## 2. Compute Returns & Momentum

Calculate daily returns and 6-month momentum factors

In [ ]:
# Calculate returns and momentum grouped by ticker
print("="*60)
print("RETURNS & MOMENTUM CALCULATION (ALL TICKERS)")
print("="*60)

# Sort by ticker and date
df_raw_sorted = df_raw.sort_index()

# Calculate returns per ticker
def calculate_returns_momentum(ticker_data):
    """Calculate returns and momentum for a single ticker"""
    ticker_data = ticker_data.sort_index()
    
    # Daily returns
    daily_ret = ticker_data['close'].pct_change()
    
    # 6-month momentum (126 trading days)
    momentum_6m = ticker_data['close'].pct_change(periods=126)
    
    return pd.DataFrame({
        'close': ticker_data['close'],
        'daily_ret': daily_ret,
        'momentum_6m': momentum_6m
    })

# Apply function to each ticker
returns_by_ticker = {}
for ticker in sorted(df_raw['ticker'].unique()):
    ticker_data = df_raw[df_raw['ticker'] == ticker]
    returns_by_ticker[ticker] = calculate_returns_momentum(ticker_data)

print(f"\n📊 Returns calculated for {len(returns_by_ticker)} tickers:")
for ticker, ret_df in returns_by_ticker.items():
    print(f"  {ticker}: {len(ret_df)} records, returns mean={ret_df['daily_ret'].mean()*100:.4f}%")

# Store for later use
print("\n✅ Returns and momentum calculated for all tickers")

RETURNS & MOMENTUM CALCULATION (ALL TICKERS)

📊 Returns calculated for 7 tickers:
  CAC: 9251 records, returns mean=0.0252%
  CCMP: 7776 records, returns mean=0.0558%
  DAX: 7305 records, returns mean=0.0375%
  IBEX: 7797 records, returns mean=0.0323%
  MIB: 7098 records, returns mean=0.0193%
  SPX: 10525 records, returns mean=0.0425%
  UKX: 10618 records, returns mean=0.0271%

✅ Returns and momentum calculated for all tickers


## 3. Signal Performance Analysis by Ticker

Select a ticker to visualize returns and momentum signals

In [ ]:
# Ticker selector and analysis with synchronized charts
from ipywidgets import widgets, Output
import plotly.graph_objects as go
from plotly.subplots import make_subplots

print("="*60)
print("TICKER SELECTION & ANALYSIS")
print("="*60)

# Create dropdown for ticker selection
available_tickers = sorted(df_raw['ticker'].unique())
ticker_dropdown = widgets.Dropdown(
    options=available_tickers,
    value=available_tickers[0],
    description='Select Ticker:',
    style={'description_width': '120px'}
)

print(f"\n📊 Available tickers: {available_tickers}")
print("\n✅ Select a ticker from the dropdown below:")
display(ticker_dropdown)

# Output area for charts
output_area = Output()
display(output_area)

# Color scheme - matches codebase dark theme
THEME_COLORS = {
    'bg': '#0b1220',
    'panel': '#0f1b33',
    'border': '#1f2d4d',
    'text': '#e6edf7',
    'muted': '#a9b7d0',
    'grid': 'rgba(31, 45, 77, 0.3)',
    'spike': 'rgba(119, 184, 255, 0.5)',
    'price': '#1f77b4',
    'positive': '#2ca02c',
    'negative': '#d62728',
    'momentum': '#9467bd',
}

def create_returns_momentum_chart(ticker_ret, ticker_name):
    """Create synchronized 3-subplot chart: price, returns, momentum"""
    daily_returns = ticker_ret['daily_ret'].copy()
    momentum_values = ticker_ret['momentum_6m'].copy()
    
    fig = make_subplots(
        rows=3, cols=1,
        subplot_titles=(f"Price - {ticker_name}", "Daily Returns (%)", "6-Month Momentum"),
        specs=[[{"secondary_y": False}], [{"secondary_y": False}], [{"secondary_y": False}]],
        vertical_spacing=0.12,
        row_heights=[0.35, 0.35, 0.3]
    )
    
    # Row 1: Price
    fig.add_trace(
        go.Scatter(
            x=ticker_ret.index, y=ticker_ret['close'],
            name='Close Price', mode='lines',
            line=dict(color=THEME_COLORS['price'], width=2),
            fill='tozeroy', fillcolor='rgba(31, 119, 180, 0.1)',
            hovertemplate='<b>%{x|%Y-%m-%d}</b><br>Price: $%{y:.2f}<extra></extra>'
        ), row=1, col=1
    )
    
    # Row 2: Returns
    colors = [THEME_COLORS['positive'] if x > 0 else THEME_COLORS['negative'] for x in daily_returns]
    fig.add_trace(
        go.Bar(
            x=ticker_ret.index, y=daily_returns * 100,
            name='Daily Return', marker=dict(color=colors), showlegend=False,
            hovertemplate='<b>%{x|%Y-%m-%d}</b><br>Return: %{y:.3f}%<extra></extra>'
        ), row=2, col=1
    )
    
    # Row 3: Momentum
    fig.add_trace(
        go.Scatter(
            x=ticker_ret.index, y=momentum_values,
            name='6M Momentum', mode='lines',
            line=dict(color=THEME_COLORS['momentum'], width=2),
            fill='tozeroy', fillcolor='rgba(148, 103, 189, 0.2)',
            hovertemplate='<b>%{x|%Y-%m-%d}</b><br>Momentum: %{y:.4f}<extra></extra>'
        ), row=3, col=1
    )
    
    fig.add_hline(y=0, line_dash="dash", line_color="rgba(0,0,0,0.3)", row=3, col=1)
    
    fig.update_xaxes(title_text="Date", row=3, col=1)
    fig.update_yaxes(title_text="Price ($)", row=1, col=1)
    fig.update_yaxes(title_text="Return (%)", row=2, col=1)
    fig.update_yaxes(title_text="Momentum", row=3, col=1)
    
    fig.update_layout(
        height=900,
        title_text=f"<b>Returns & Momentum Analysis - {ticker_name}</b>",
        hovermode='x unified',
        plot_bgcolor=THEME_COLORS['panel'],
        paper_bgcolor=THEME_COLORS['bg'],
        font=dict(family="Arial, sans-serif", size=11, color=THEME_COLORS['text']),
        margin=dict(l=70, r=30, t=80, b=60),
        showlegend=False,
        title_font=dict(size=14, color=THEME_COLORS['text'])
    )
    
    fig.update_xaxes(
        gridcolor=THEME_COLORS['grid'],
        tickfont=dict(color=THEME_COLORS['muted']),
        showspikes=True, spikemode='across', spikethickness=1.5,
        spikecolor=THEME_COLORS['spike']
    )
    fig.update_yaxes(
        gridcolor=THEME_COLORS['grid'],
        tickfont=dict(color=THEME_COLORS['muted']),
        showspikes=True, spikethickness=1.5,
        spikecolor=THEME_COLORS['spike']
    )
    
    return fig

def update_chart(change):
    """Update chart when dropdown changes"""
    with output_area:
        output_area.clear_output(wait=True)
        selected_ticker = ticker_dropdown.value
        
        ticker_ret = returns_by_ticker[selected_ticker].copy()
        ticker_ret['momentum_6m'] = ticker_ret['close'].pct_change(periods=126)
        
        print(f"\n📊 Analyzing: {selected_ticker}")
        print(f"  Records: {len(ticker_ret)}")
        print(f"  Date range: {ticker_ret.index.min().date()} to {ticker_ret.index.max().date()}")
        
        daily_returns = ticker_ret['daily_ret'].copy()
        print(f"\n📈 Daily Returns:")
        print(f"  Mean: {daily_returns.mean()*100:.4f}%")
        print(f"  Std Dev: {daily_returns.std()*100:.4f}%")
        print(f"  Sharpe (annualized): {daily_returns.mean()/daily_returns.std() * np.sqrt(252):.3f}")
        
        momentum_values = ticker_ret['momentum_6m'].copy()
        print(f"\n💹 6-Month Momentum:")
        print(f"  Mean: {momentum_values.mean():.4f}")
        print(f"  Std Dev: {momentum_values.std():.4f}")
        
        fig = create_returns_momentum_chart(ticker_ret, selected_ticker)
        print("\n✅ Displaying synchronized analysis charts")
        fig.show()

ticker_dropdown.observe(update_chart, names='value')
update_chart(None)

## 4. Signal Generation & Cross-Sectional Ranking

Generate momentum signals at multiple horizons and rank tickers by signal strength

In [ ]:
# Calculate momentum signals at multiple horizons
print("="*60)
print("SIGNAL GENERATION - MULTIPLE HORIZONS")
print("="*60)

# Momentum windows (trading days)
momentum_windows = {
    'mom_1m': 21,      # 1 month
    'mom_3m': 63,      # 3 months (126 trading days = 6 months)
    'mom_6m': 126,     # 6 months
    'mom_9m': 189,     # 9 months
    'mom_12m': 252,    # 12 months (1 year)
}

# Calculate signals for all tickers
signals_by_ticker = {}
for ticker, ticker_ret in returns_by_ticker.items():
    signals = ticker_ret[['close']].copy()
    
    # Calculate momentum for each window
    for signal_name, window in momentum_windows.items():
        signals[signal_name] = ticker_ret['close'].pct_change(periods=window)
    
    signals_by_ticker[ticker] = signals

print(f"\n📊 Signals calculated for {len(signals_by_ticker)} tickers")
print(f"📈 Momentum horizons: {list(momentum_windows.keys())}")
print(f"✅ Signals ready for cross-sectional ranking")

# Display sample signals for first ticker
first_ticker = list(signals_by_ticker.keys())[0]
print(f"\n🔍 Sample signals ({first_ticker}):")
display(signals_by_ticker[first_ticker].iloc[-5:])


SIGNAL GENERATION - MULTIPLE HORIZONS

📊 Signals calculated for 7 tickers
📈 Momentum horizons: ['mom_1m', 'mom_3m', 'mom_6m', 'mom_9m', 'mom_12m']
✅ Signals ready for cross-sectional ranking

🔍 Sample signals (CAC):


,close,mom_1m,mom_3m,mom_6m,mom_9m,mom_12m
date,,,,,,
2025-12-08,8108.43,0.019905,0.044723,0.055142,0.021443,0.092280
2025-12-09,8052.51,-0.000372,0.029269,0.040075,0.003018,0.085106
2025-12-10,8022.69,-0.016373,0.025232,0.044114,-0.006353,0.082747
2025-12-11,8085.76,-0.018866,0.023912,0.056117,-0.003550,0.099045
2025-12-12,8068.62,-0.019905,0.032028,0.068203,-0.012586,0.095432


In [ ]:
# Cross-sectional ranking of signals
print("="*60)
print("CROSS-SECTIONAL RANKING")
print("="*60)

# Combine all signals into one DataFrame with hierarchical index (date, ticker)
all_signals_list = []
for ticker, signals_df in signals_by_ticker.items():
    signals_df = signals_df.copy()
    signals_df['ticker'] = ticker
    all_signals_list.append(signals_df)

all_signals = pd.concat(all_signals_list)
all_signals = all_signals.reset_index()
all_signals.columns = ['date', 'close', 'mom_1m', 'mom_3m', 'mom_6m', 'mom_9m', 'mom_12m', 'ticker']
all_signals = all_signals.set_index(['date', 'ticker']).sort_index()

print(f"\n📊 Combined signals shape: {all_signals.shape}")
print(f"   Dates: {all_signals.index.get_level_values('date').min()} to {all_signals.index.get_level_values('date').max()}")
print(f"   Tickers: {sorted(all_signals.index.get_level_values('ticker').unique())}")

# Calculate cross-sectional ranks (percentile ranks)
# Rank on each date independently - higher rank = stronger signal
def rank_signals(signal_df, signal_col):
    """Rank tickers on a given date by signal strength (higher = stronger)"""
    return signal_df[signal_col].rank(pct=True, method='average')

ranks_by_date_signal = {}
for signal_col in ['mom_1m', 'mom_3m', 'mom_6m', 'mom_9m', 'mom_12m']:
    ranks = all_signals.groupby(level='date').apply(lambda x: rank_signals(x, signal_col))
    ranks.name = f'{signal_col}_rank'
    ranks_by_date_signal[signal_col] = ranks

# Add ranks to signals DataFrame
for signal_col, ranks in ranks_by_date_signal.items():
    all_signals[f'{signal_col}_rank'] = ranks.values

print(f"\n✅ Cross-sectional rankings calculated")

# Show ranking distribution for latest date
latest_date = all_signals.index.get_level_values('date').max()
latest_rankings = all_signals.loc[latest_date, [col for col in all_signals.columns if 'rank' in col]]

print(f"\n📈 Latest rankings ({latest_date.date()}):")
display(latest_rankings)


CROSS-SECTIONAL RANKING

📊 Combined signals shape: (60370, 6)
   Dates: 1983-12-30 00:00:00 to 2025-12-12 00:00:00
   Tickers: ['CAC', 'CCMP', 'DAX', 'IBEX', 'MIB', 'SPX', 'UKX']

✅ Cross-sectional rankings calculated

📈 Latest rankings (2025-12-12):


,mom_1m_rank,mom_3m_rank,mom_6m_rank,mom_9m_rank,mom_12m_rank
ticker,,,,,
CAC,0.285714,0.285714,0.285714,0.142857,0.142857
CCMP,0.571429,0.714286,0.857143,1.000000,0.571429
DAX,0.857143,0.571429,0.142857,0.285714,0.714286
IBEX,1.000000,1.000000,1.000000,0.857143,1.000000
MIB,0.142857,0.142857,0.571429,0.428571,0.857143
SPX,0.714286,0.428571,0.714286,0.714286,0.285714
UKX,0.428571,0.857143,0.428571,0.571429,0.428571


In [ ]:
# Visualize signal rankings heatmap
print("="*60)
print("SIGNAL RANKINGS HEATMAP")
print("="*60)

# Prepare ranking data for heatmap (last 5 years)
five_years_ago = all_signals.index.get_level_values('date').max() - pd.Timedelta(days=252*5)
recent_signals = all_signals[all_signals.index.get_level_values('date') >= five_years_ago]

# Reshape rankings for heatmap: rows = dates, columns = tickers, values = 6m momentum rank
mom_6m_rankings = recent_signals['mom_6m_rank'].unstack(fill_value=np.nan)

# Create heatmap
fig_heatmap = go.Figure(data=go.Heatmap(
    z=mom_6m_rankings.values,
    x=mom_6m_rankings.columns,
    y=mom_6m_rankings.index,
    colorscale='RdYlGn',
    zmid=0.5,
    colorbar=dict(title="Rank<br>(Percentile)")
))

fig_heatmap.update_layout(
    title='<b>6-Month Momentum Rankings - Last 5 Years</b>',
    xaxis_title='Ticker',
    yaxis_title='Date',
    height=600,
    plot_bgcolor=THEME_COLORS['panel'],
    paper_bgcolor=THEME_COLORS['bg'],
    font=dict(family="Arial, sans-serif", size=10, color=THEME_COLORS['text']),
    title_font=dict(size=14, color=THEME_COLORS['text'])
)

fig_heatmap.update_xaxes(tickfont=dict(color=THEME_COLORS['muted']))
fig_heatmap.update_yaxes(tickfont=dict(color=THEME_COLORS['muted']))

print("✅ Displaying momentum rankings heatmap")
fig_heatmap.show()

print(f"\n📊 Ranking statistics (6-month momentum):")
print(f"   Mean rank: {mom_6m_rankings.mean().mean():.3f}")
print(f"   Std dev: {mom_6m_rankings.std().std():.3f}")


SIGNAL RANKINGS HEATMAP
✅ Displaying momentum rankings heatmap



📊 Ranking statistics (6-month momentum):
   Mean rank: 0.573
   Std dev: 0.047


In [ ]:
# Long/Short portfolio construction based on ranks
print("="*60)
print("LONG/SHORT PORTFOLIO CONSTRUCTION")
print("="*60)

# Portfolio size for position value calculations
PORTFOLIO_SIZE = 500_000

# Create quantile-based long/short signals
# Top 50% by rank = Long (rank > 0.5)
# Bottom 50% by rank = Short (rank <= 0.5)

def create_portfolio_signals(rankings_df, quantile_threshold=0.5):
    """Create long/short portfolio signals based on ranking quantiles"""
    portfolio_signal = rankings_df.copy()
    portfolio_signal[:] = 0
    portfolio_signal[rankings_df > quantile_threshold] = 1.0   # Long
    portfolio_signal[rankings_df <= quantile_threshold] = -1.0  # Short
    return portfolio_signal

# Create long/short signals for 6-month momentum
ls_signals = create_portfolio_signals(mom_6m_rankings, quantile_threshold=0.5)

print(f"\n📊 Long/Short signal construction:")
print(f"   Quantile threshold: 0.5 (top 50% vs bottom 50%)")
print(f"   Long signal: 1.0, Short signal: -1.0")
print(f"   Portfolio size: ${PORTFOLIO_SIZE:,}")
print(f"   Position allocation: 50% to longs, 50% to shorts")

# Calculate portfolio returns for each date
ls_returns = []
dates = []
all_dates = sorted(ls_signals.index.tolist())

for i in range(len(all_dates) - 1):
    curr_date = all_dates[i]
    next_date = all_dates[i + 1]
    
    try:
        # Get signals for current date
        signal_row = ls_signals.loc[curr_date]
        long_tickers = signal_row[signal_row == 1.0].index.tolist()
        short_tickers = signal_row[signal_row == -1.0].index.tolist()
        
        if not (long_tickers or short_tickers):
            continue
        
        # Calculate position values (actual dollars deployed per position)
        long_count = len(long_tickers)
        short_count = len(short_tickers)
        
        long_position_value = (PORTFOLIO_SIZE * 0.50) / long_count if long_count > 0 else 0
        short_position_value = (PORTFOLIO_SIZE * 0.50) / short_count if short_count > 0 else 0
        
        # Accumulate portfolio P&L in dollars
        portfolio_pnl_dollars = 0.0
        
        # Process long positions (profit when underlying goes UP)
        for ticker in long_tickers:
            try:
                ret = returns_by_ticker[ticker].loc[next_date, 'daily_ret']
                if not np.isnan(ret):
                    ticker_pnl = long_position_value * ret
                    portfolio_pnl_dollars += ticker_pnl
            except:
                pass
        
        # Process short positions (profit when underlying goes DOWN, so negate return)
        for ticker in short_tickers:
            try:
                ret = returns_by_ticker[ticker].loc[next_date, 'daily_ret']
                if not np.isnan(ret):
                    ticker_pnl = short_position_value * (-ret)  # Negate: short profits on down moves
                    portfolio_pnl_dollars += ticker_pnl
            except:
                pass
        
        # Convert dollar P&L to portfolio return (as percentage of PORTFOLIO_SIZE)
        if portfolio_pnl_dollars != 0:
            ls_pnl = portfolio_pnl_dollars / PORTFOLIO_SIZE
            ls_returns.append(ls_pnl)
            dates.append(curr_date)
    except:
        pass

# Create performance series
ls_performance = pd.Series(ls_returns, index=dates)
ls_cumulative = (1 + ls_performance).cumprod() - 1

print(f"\n✅ Long/Short forward returns calculated (POSITION-WEIGHTED)")
print(f"   Observations: {len(ls_performance)}")
if len(ls_performance) > 0:
    print(f"   Mean daily return: {ls_performance.mean()*100:.4f}%")
    print(f"   Std dev: {ls_performance.std()*100:.4f}%")
    print(f"   Sharpe ratio (annualized): {ls_performance.mean()/ls_performance.std() * np.sqrt(252):.3f}")
    print(f"   Cumulative return: {ls_cumulative.iloc[-1]*100:.2f}%")

# Plot cumulative L-S returns
fig_ls = go.Figure()

fig_ls.add_trace(go.Scatter(
    x=ls_cumulative.index,
    y=ls_cumulative.values * 100,
    name='L-S Cumulative Return',
    mode='lines',
    line=dict(color='#22d3ee', width=2),
    fill='tozeroy',
    fillcolor='rgba(34, 211, 238, 0.1)',
    hovertemplate='<b>%{x|%Y-%m-%d}</b><br>Return: %{y:.2f}%<extra></extra>'
))

fig_ls.update_layout(
    title='<b>Long/Short Portfolio Cumulative Return (Position-Weighted by Dollar Value)</b>',
    xaxis_title='Date',
    yaxis_title='Cumulative Return (%)',
    height=500,
    hovermode='x unified',
    plot_bgcolor=THEME_COLORS['panel'],
    paper_bgcolor=THEME_COLORS['bg'],
    font=dict(family="Arial, sans-serif", size=11, color=THEME_COLORS['text']),
    margin=dict(l=70, r=30, t=80, b=60),
    title_font=dict(size=14, color=THEME_COLORS['text'])
)

fig_ls.update_xaxes(
    gridcolor=THEME_COLORS['grid'],
    tickfont=dict(color=THEME_COLORS['muted'])
)
fig_ls.update_yaxes(
    gridcolor=THEME_COLORS['grid'],
    tickfont=dict(color=THEME_COLORS['muted'])
)

print("\n✅ Displaying Long/Short cumulative performance")
fig_ls.show()

LONG/SHORT PORTFOLIO CONSTRUCTION

📊 Long/Short signal construction:
   Quantile threshold: 0.5 (top 50% vs bottom 50%)
   Long signal: 1.0, Short signal: -1.0
   Portfolio size: $500,000
   Position allocation: 50% to longs, 50% to shorts

✅ Long/Short forward returns calculated (POSITION-WEIGHTED)
   Observations: 892
   Mean daily return: 0.0025%
   Std dev: 0.3438%
   Sharpe ratio (annualized): 0.113
   Cumulative return: 1.67%

✅ Displaying Long/Short cumulative performance


In [ ]:
# Position tracking per ticker and performance decomposition
print("="*60)
print("POSITION TRACKING & PER-TICKER PERFORMANCE")
print("="*60)

# Track positions for each ticker over time
positions_by_ticker = {}
ticker_returns = {}
ticker_pnl = {}

for ticker in available_tickers:
    positions_by_ticker[ticker] = []
    ticker_returns[ticker] = []
    ticker_pnl[ticker] = []

# Track dates
all_backtest_dates = []

# Loop through all dates
all_unique_dates = sorted(ls_signals.index.tolist())

for i in range(len(all_unique_dates) - 1):
    curr_date = all_unique_dates[i]
    next_date = all_unique_dates[i + 1]
    
    try:
        # Get signals for current date
        signal_row = ls_signals.loc[curr_date]
        
        # Track positions and returns for each ticker
        for ticker in available_tickers:
            position = signal_row.get(ticker, 0)  # 1.0 = long, -1.0 = short, 0 = neutral
            positions_by_ticker[ticker].append(position)
            
            # Get next-day return for this ticker
            try:
                ret = returns_by_ticker[ticker].loc[next_date, 'daily_ret']
                if np.isnan(ret):
                    ret = 0
            except:
                ret = 0
            
            ticker_returns[ticker].append(ret)
            
            # Calculate PnL for this position
            # PnL = position * return (long position benefits from positive returns, short benefits from negative)
            pnl = position * ret
            ticker_pnl[ticker].append(pnl)
        
        all_backtest_dates.append(curr_date)
    except Exception as e:
        pass

print(f"\n✅ Position tracking completed")
print(f"   Backtest dates: {len(all_backtest_dates)} observations")
print(f"   Tickers tracked: {available_tickers}")

# Convert to DataFrames for analysis
positions_df = pd.DataFrame(positions_by_ticker, index=all_backtest_dates)
returns_df = pd.DataFrame(ticker_returns, index=all_backtest_dates)
pnl_df = pd.DataFrame(ticker_pnl, index=all_backtest_dates)

# Calculate cumulative PnL per ticker
cumulative_pnl_by_ticker = {}
for ticker in available_tickers:
    cumulative_pnl_by_ticker[ticker] = (1 + pnl_df[ticker]).cumprod() - 1

cumulative_pnl_df = pd.DataFrame(cumulative_pnl_by_ticker, index=all_backtest_dates)

# Summary statistics per ticker
print(f"\n📊 PER-TICKER PERFORMANCE SUMMARY:")
print("-" * 60)
perf_summary = []
for ticker in available_tickers:
    mean_ret = pnl_df[ticker].mean()
    std_ret = pnl_df[ticker].std()
    sharpe = mean_ret / std_ret * np.sqrt(252) if std_ret > 0 else 0
    cumret = cumulative_pnl_df[ticker].iloc[-1]
    
    perf_summary.append({
        'Ticker': ticker,
        'Mean Daily PnL': f"{mean_ret*100:.4f}%",
        'Std Dev': f"{std_ret*100:.4f}%",
        'Sharpe': f"{sharpe:.3f}",
        'Cum Return': f"{cumret*100:.2f}%",
        'Avg Position': f"{positions_df[ticker].mean():.2f}",
        'Long Days': int((positions_df[ticker] > 0).sum()),
        'Short Days': int((positions_df[ticker] < 0).sum()),
    })

perf_summary_df = pd.DataFrame(perf_summary)
display(perf_summary_df)

print(f"\n✅ Per-ticker analysis ready")


POSITION TRACKING & PER-TICKER PERFORMANCE

✅ Position tracking completed
   Backtest dates: 892 observations
   Tickers tracked: ['CAC', 'CCMP', 'DAX', 'IBEX', 'MIB', 'SPX', 'UKX']

📊 PER-TICKER PERFORMANCE SUMMARY:
------------------------------------------------------------


,Ticker,Mean Daily PnL,Std Dev,Sharpe,Cum Return,Avg Position,Long Days,Short Days
0,CAC,0.0076%,0.9194%,0.132,3.10%,-0.39,268,616
1,CCMP,-0.0015%,1.3428%,-0.017,-9.00%,0.26,549,317
2,DAX,0.0461%,0.9559%,0.766,44.86%,0.33,590,292
3,IBEX,0.0098%,0.9267%,0.168,5.03%,0.55,689,195
4,MIB,0.0307%,1.0437%,0.466,25.21%,0.45,638,240
5,SPX,0.0173%,1.0197%,0.270,11.39%,0.20,523,343
6,UKX,-0.0243%,0.6968%,-0.553,-21.19%,-0.43,244,629



✅ Per-ticker analysis ready


In [ ]:
# Per-ticker signals and performance with interactive toggle
print("="*60)
print("PER-TICKER SIGNALS & PERFORMANCE ANALYSIS")
print("="*60)

from ipywidgets import widgets, HBox, VBox

# INTERACTIVE SIGNALS & PnL CHARTS
print("\n✅ Creating interactive signal and performance analysis...\n")

# Create checkbox for each ticker
ticker_checkboxes = {}
for ticker in available_tickers:
    ticker_checkboxes[ticker] = widgets.Checkbox(
        value=True,
        description=ticker,
        indent=False
    )

# Button to toggle all on/off
toggle_all_button = widgets.Button(description='Toggle All')
toggle_all_state = {'value': True}

def toggle_all_clicked(b):
    toggle_all_state['value'] = not toggle_all_state['value']
    for ticker in available_tickers:
        ticker_checkboxes[ticker].value = toggle_all_state['value']

toggle_all_button.on_click(toggle_all_clicked)

# Output area for chart
perf_output_area = widgets.Output()

def update_perf_chart(change=None):
    """Update performance chart based on checkbox selections"""
    with perf_output_area:
        perf_output_area.clear_output(wait=True)
        
        # Determine which tickers to show
        selected_tickers = [t for t in available_tickers if ticker_checkboxes[t].value]
        
        if not selected_tickers:
            print("⚠️ Select at least one ticker to display")
            return
        
        # Calculate position values in dollars for each date
        # Position value = signal * (50% / count) * portfolio_size
        position_values = {}
        for ticker in available_tickers:
            values = []
            for date_idx, date in enumerate(positions_df.index):
                signal = positions_df.loc[date, ticker]
                
                if signal == 0:
                    values.append(0)
                elif signal > 0:  # Long position
                    long_count = (positions_df.loc[date] > 0).sum()
                    if long_count > 0:
                        allocation = (PORTFOLIO_SIZE * 0.50) / long_count
                        values.append(allocation)  # Positive for long
                    else:
                        values.append(0)
                else:  # Short position (signal < 0)
                    short_count = (positions_df.loc[date] < 0).sum()
                    if short_count > 0:
                        allocation = (PORTFOLIO_SIZE * 0.50) / short_count
                        values.append(-allocation)  # Negative for short
                    else:
                        values.append(0)
            
            position_values[ticker] = values
        
        position_values_df = pd.DataFrame(position_values, index=positions_df.index)
        
        # Create subplots with signals on top, cumulative PnL below
        fig_perf = make_subplots(
            rows=3, cols=1,
            subplot_titles=("Trading Signal (+1.0=Long, -1.0=Short)", 
                           "Position Value in $ (+Long / -Short)", 
                           "Cumulative PnL by Ticker"),
            vertical_spacing=0.12,
            row_heights=[0.3, 0.3, 0.4]
        )
        
        # Add signal traces (top chart)
        for ticker in selected_tickers:
            fig_perf.add_trace(
                go.Scatter(
                    x=positions_df.index,
                    y=positions_df[ticker],
                    name=ticker,
                    mode='lines',
                    line=dict(width=2),
                    hovertemplate=f'<b>{ticker}</b><br>%{{x|%Y-%m-%d}}<br>Signal: %{{y:.2f}}<extra></extra>',
                    legendgroup=ticker,
                    showlegend=True
                ),
                row=1, col=1
            )
        
        # Add reference lines for signals
        fig_perf.add_hline(y=0, line_dash="dash", line_color="rgba(255,255,255,0.2)", row=1, col=1)
        fig_perf.add_hline(y=1, line_dash="dot", line_color="rgba(44, 160, 44, 0.3)", row=1, col=1)
        fig_perf.add_hline(y=-1, line_dash="dot", line_color="rgba(214, 39, 40, 0.3)", row=1, col=1)
        
        # Add position value traces (middle chart)
        for ticker in selected_tickers:
            fig_perf.add_trace(
                go.Scatter(
                    x=position_values_df.index,
                    y=position_values_df[ticker],
                    name=ticker,
                    mode='lines',
                    line=dict(width=2),
                    fill='tozeroy',
                    hovertemplate='<b>' + ticker + '</b><br>%{x|%Y-%m-%d}<br>Position: $%{y:,.0f}<extra></extra>',
                    legendgroup=ticker,
                    showlegend=False
                ),
                row=2, col=1
            )
        
        # Add zero line for position value
        fig_perf.add_hline(y=0, line_dash="dash", line_color="rgba(255,255,255,0.2)", row=2, col=1)
        
        # Add cumulative PnL traces (bottom chart)
        for ticker in selected_tickers:
            fig_perf.add_trace(
                go.Scatter(
                    x=cumulative_pnl_df.index,
                    y=cumulative_pnl_df[ticker] * 100,
                    name=ticker,
                    mode='lines',
                    line=dict(width=2),
                    hovertemplate=f'<b>{ticker}</b><br>%{{x|%Y-%m-%d}}<br>Return: %{{y:.2f}}%<extra></extra>',
                    legendgroup=ticker,
                    showlegend=False
                ),
                row=3, col=1
            )
        
        # Add zero line for PnL
        fig_perf.add_hline(y=0, line_dash="dash", line_color="rgba(255,255,255,0.2)", row=3, col=1)
        
        # Update axes
        fig_perf.update_xaxes(title_text="Date", row=3, col=1)
        fig_perf.update_yaxes(title_text="Signal", row=1, col=1)
        fig_perf.update_yaxes(title_text="Position Value ($)", row=2, col=1)
        fig_perf.update_yaxes(title_text="Return (%)", row=3, col=1)
        
        fig_perf.update_layout(
            title='<b>Per-Ticker Signals, Position Values & Cumulative PnL</b>',
            height=1200,
            hovermode='x unified',
            plot_bgcolor=THEME_COLORS['panel'],
            paper_bgcolor=THEME_COLORS['bg'],
            font=dict(family="Arial, sans-serif", size=11, color=THEME_COLORS['text']),
            margin=dict(l=100, r=30, t=80, b=60),
            title_font=dict(size=14, color=THEME_COLORS['text']),
            legend=dict(bgcolor='rgba(31, 45, 77, 0.5)', bordercolor=THEME_COLORS['border'], borderwidth=1)
        )
        
        fig_perf.update_xaxes(
            gridcolor=THEME_COLORS['grid'],
            tickfont=dict(color=THEME_COLORS['muted'])
        )
        fig_perf.update_yaxes(
            gridcolor=THEME_COLORS['grid'],
            tickfont=dict(color=THEME_COLORS['muted'])
        )
        
        fig_perf.show()

# Connect checkboxes to update function
for ticker in available_tickers:
    ticker_checkboxes[ticker].observe(update_perf_chart, names='value')

# Layout
checkbox_layout = HBox([ticker_checkboxes[t] for t in available_tickers[:4]])
checkbox_layout2 = HBox([ticker_checkboxes[t] for t in available_tickers[4:]])
controls = VBox([
    widgets.HTML("<b>Select Tickers to Display:</b>"),
    checkbox_layout,
    checkbox_layout2,
    toggle_all_button
])

display(controls)
display(perf_output_area)

# Initial display
update_perf_chart()

print("✅ Per-ticker performance analysis ready - toggle checkboxes to show/hide tickers")


In [ ]:
# Detailed position history and contribution analysis
print("="*60)
print("POSITION HISTORY & CONTRIBUTION ANALYSIS")
print("="*60)

# 1. Position transition table
print("\n📊 POSITION TRANSITIONS BY TICKER:")
print("-" * 80)

position_transitions = []
for ticker in available_tickers:
    long_count = (positions_df[ticker] > 0).sum()
    short_count = (positions_df[ticker] < 0).sum()
    neutral_count = (positions_df[ticker] == 0).sum()
    
    position_transitions.append({
        'Ticker': ticker,
        'Long %': f"{100*long_count/len(positions_df):.1f}%",
        'Short %': f"{100*short_count/len(positions_df):.1f}%",
        'Neutral %': f"{100*neutral_count/len(positions_df):.1f}%",
    })

trans_df = pd.DataFrame(position_transitions)
display(trans_df)

# 2. Correlation of returns
print("\n🔗 TICKER RETURN CORRELATIONS:")
print("-" * 80)
ticker_corr = returns_df.corr()
display(ticker_corr)

# 3. Contribution to overall P&L
print("\n💰 CONTRIBUTION TO OVERALL L/S P&L:")
print("-" * 80)

# Calculate cumulative contribution per ticker
contributions = []
total_pnl = pnl_df.sum().sum()

for ticker in available_tickers:
    ticker_total_pnl = pnl_df[ticker].sum()
    ticker_contribution = ticker_total_pnl / total_pnl if total_pnl != 0 else 0
    
    contributions.append({
        'Ticker': ticker,
        'Total PnL': f"${ticker_total_pnl*10000:.2f}",  # Scale for readability
        'Contribution %': f"{ticker_contribution*100:.1f}%",
        'Avg Daily PnL': f"{pnl_df[ticker].mean()*100:.4f}%",
        'Win Rate': f"{(pnl_df[ticker] > 0).sum() / len(pnl_df[ticker]) * 100:.1f}%"
    })

contrib_df = pd.DataFrame(contributions)
display(contrib_df)

# 4. Comparative performance chart
print("\n📈 Creating comparative performance dashboard...\n")

fig_compare = make_subplots(
    rows=2, cols=2,
    subplot_titles=("Cumulative Return by Ticker", "Win Rate Distribution", 
                   "Sharpe Ratio by Ticker", "Position Exposure Over Time"),
    specs=[[{"secondary_y": False}, {"secondary_y": False}],
           [{"secondary_y": False}, {"secondary_y": False}]]
)

# Row 1, Col 1: Cumulative returns
for ticker in available_tickers:
    fig_compare.add_trace(
        go.Scatter(x=cumulative_pnl_df.index, y=cumulative_pnl_df[ticker] * 100,
                   name=ticker, mode='lines'),
        row=1, col=1
    )

# Row 1, Col 2: Win rate bar chart
win_rates = [(pnl_df[ticker] > 0).sum() / len(pnl_df[ticker]) * 100 for ticker in available_tickers]
fig_compare.add_trace(
    go.Bar(x=available_tickers, y=win_rates, name='Win Rate %', marker_color='rgba(100, 200, 100, 0.7)',
           hovertemplate='<b>%{x}</b><br>Win Rate: %{y:.1f}%<extra></extra>'),
    row=1, col=2
)

# Row 2, Col 1: Sharpe ratios
sharpe_ratios = []
for ticker in available_tickers:
    mean_ret = pnl_df[ticker].mean()
    std_ret = pnl_df[ticker].std()
    sharpe = mean_ret / std_ret * np.sqrt(252) if std_ret > 0 else 0
    sharpe_ratios.append(sharpe)

colors_sharpe = ['green' if s > 0 else 'red' for s in sharpe_ratios]
fig_compare.add_trace(
    go.Bar(x=available_tickers, y=sharpe_ratios, name='Sharpe Ratio', marker_color=colors_sharpe,
           hovertemplate='<b>%{x}</b><br>Sharpe: %{y:.3f}<extra></extra>'),
    row=2, col=1
)

# Row 2, Col 2: Net position exposure over time
net_exposure = (positions_df > 0).sum(axis=1) - (positions_df < 0).sum(axis=1)
fig_compare.add_trace(
    go.Scatter(x=positions_df.index, y=net_exposure, name='Net Exposure',
               mode='lines', fill='tozeroy', line=dict(color='#22d3ee', width=2),
               hovertemplate='<b>%{x|%Y-%m-%d}</b><br>Net Longs - Shorts: %{y:.0f}<extra></extra>'),
    row=2, col=2
)

# Update axes labels
fig_compare.update_xaxes(title_text="Date", row=1, col=1)
fig_compare.update_yaxes(title_text="Cumulative Return (%)", row=1, col=1)
fig_compare.update_xaxes(title_text="Ticker", row=1, col=2)
fig_compare.update_yaxes(title_text="Win Rate (%)", row=1, col=2)
fig_compare.update_xaxes(title_text="Ticker", row=2, col=1)
fig_compare.update_yaxes(title_text="Sharpe Ratio", row=2, col=1)
fig_compare.update_xaxes(title_text="Date", row=2, col=2)
fig_compare.update_yaxes(title_text="# of Long Positions - # of Short", row=2, col=2)

# Update layout
fig_compare.update_layout(
    title='<b>Comprehensive Performance Dashboard</b>',
    height=900,
    plot_bgcolor=THEME_COLORS['panel'],
    paper_bgcolor=THEME_COLORS['bg'],
    font=dict(family="Arial, sans-serif", size=10, color=THEME_COLORS['text']),
    title_font=dict(size=14, color=THEME_COLORS['text']),
    showlegend=True,
    hovermode='closest'
)

fig_compare.update_xaxes(gridcolor=THEME_COLORS['grid'], tickfont=dict(color=THEME_COLORS['muted']))
fig_compare.update_yaxes(gridcolor=THEME_COLORS['grid'], tickfont=dict(color=THEME_COLORS['muted']))

print("✅ Displaying comprehensive performance dashboard")
fig_compare.show()

print("\n✅ Position history and contribution analysis complete")


POSITION HISTORY & CONTRIBUTION ANALYSIS

📊 POSITION TRANSITIONS BY TICKER:
--------------------------------------------------------------------------------


,Ticker,Long %,Short %,Neutral %
0,CAC,30.0%,69.1%,0.9%
1,CCMP,61.5%,35.5%,2.9%
2,DAX,66.1%,32.7%,1.1%
3,IBEX,77.2%,21.9%,0.9%
4,MIB,71.5%,26.9%,1.6%
5,SPX,58.6%,38.5%,2.9%
6,UKX,27.4%,70.5%,2.1%



🔗 TICKER RETURN CORRELATIONS:
--------------------------------------------------------------------------------


,CAC,CCMP,DAX,IBEX,MIB,SPX,UKX
CAC,1.000000,0.332107,0.875300,0.766021,0.852961,0.375973,0.745352
CCMP,0.332107,1.000000,0.384661,0.275488,0.330441,0.963499,0.214709
DAX,0.875300,0.384661,1.000000,0.790170,0.861388,0.414354,0.716385
IBEX,0.766021,0.275488,0.790170,1.000000,0.819466,0.332699,0.722837
MIB,0.852961,0.330441,0.861388,0.819466,1.000000,0.377801,0.713771
SPX,0.375973,0.963499,0.414354,0.332699,0.377801,1.000000,0.291396
UKX,0.745352,0.214709,0.716385,0.722837,0.713771,0.291396,1.000000



💰 CONTRIBUTION TO OVERALL L/S P&L:
--------------------------------------------------------------------------------


,Ticker,Total PnL,Contribution %,Avg Daily PnL,Win Rate
0,CAC,$681.43,8.9%,0.0076%,48.9%
1,CCMP,$-131.78,-1.7%,-0.0015%,50.6%
2,DAX,$4114.11,53.7%,0.0461%,50.0%
3,IBEX,$874.61,11.4%,0.0098%,52.4%
4,MIB,$2735.67,35.7%,0.0307%,52.1%
5,SPX,$1545.50,20.2%,0.0173%,50.1%
6,UKX,$-2163.67,-28.3%,-0.0243%,47.2%



📈 Creating comparative performance dashboard...

✅ Displaying comprehensive performance dashboard



✅ Position history and contribution analysis complete


In [ ]:

# Portfolio value evolution and returns analysis
print("="*80)
print("PORTFOLIO VALUE & RETURNS ANALYSIS")
print("="*80)

# Portfolio size ($500K - margin account, capital deployed: 50% long + 50% short = 100% leverage)
PORTFOLIO_SIZE = 500_000

# Calculate daily portfolio returns from L-S strategy
portfolio_returns = ls_performance.copy()

# Calculate portfolio value over time
portfolio_value = PORTFOLIO_SIZE * (1 + portfolio_returns).cumprod()

# Summary statistics
initial_value = PORTFOLIO_SIZE
final_value = portfolio_value.iloc[-1]
total_return_dollars = final_value - initial_value
total_return_pct = (final_value / initial_value - 1) * 100

# Daily statistics
mean_daily_ret = portfolio_returns.mean()
std_daily_ret = portfolio_returns.std()
sharpe_ratio = mean_daily_ret / std_daily_ret * np.sqrt(252) if std_daily_ret > 0 else 0

# Drawdown analysis
cumulative_returns = (1 + portfolio_returns).cumprod()
running_max = cumulative_returns.expanding().max()
drawdown = (cumulative_returns - running_max) / running_max
max_drawdown = drawdown.min()

print(f"""
💰 PORTFOLIO PERFORMANCE SUMMARY (Starting with ${PORTFOLIO_SIZE:,.0f})
═════════════════════════════════════════════════════════════════════════════

FINAL RESULTS:
  Initial Portfolio Value:  ${initial_value:>15,.0f}
  Final Portfolio Value:    ${final_value:>15,.0f}
  Total Return ($):         ${total_return_dollars:>15,.0f}
  Total Return (%):         {total_return_pct:>15.2f}%

PERFORMANCE METRICS:
  Mean Daily Return:        {mean_daily_ret*100:>15.4f}%
  Daily Std Deviation:      {std_daily_ret*100:>15.4f}%
  Sharpe Ratio (annual):    {sharpe_ratio:>15.3f}
  Max Drawdown:             {max_drawdown*100:>15.2f}%
  
  Backtest Period:          {portfolio_returns.index.min().date()} to {portfolio_returns.index.max().date()}
  Trading Days:             {len(portfolio_returns):>15,.0f}

═════════════════════════════════════════════════════════════════════════════
""")

# Monthly returns table
portfolio_value_df = pd.DataFrame({
    'Portfolio_Value': portfolio_value,
    'Daily_Return': portfolio_returns,
})

# Extract month-end values
monthly_values = portfolio_value_df['Portfolio_Value'].resample('M').last()
monthly_returns = portfolio_value_df['Daily_Return'].resample('M').sum()

print("\n📅 MONTHLY RETURNS (Year-Month: Return $):")
print("-" * 80)
monthly_summary = []
for date, ret in monthly_returns.items():
    ret_dollars = monthly_values.loc[date] - monthly_values.shift(1).loc[date]
    monthly_summary.append({
        'Date': date.strftime('%Y-%m'),
        'Return $': ret_dollars,
        'Return %': ret * 100,
        'Portfolio Value': monthly_values.loc[date]
    })

monthly_df = pd.DataFrame(monthly_summary)
display(monthly_df.tail(12))  # Show last 12 months

# Create portfolio value chart
print("\n📈 Creating portfolio value chart over time...\n")

fig_portfolio = go.Figure()

# Portfolio value line
fig_portfolio.add_trace(go.Scatter(
    x=portfolio_value.index,
    y=portfolio_value.values,
    name='Portfolio Value',
    mode='lines',
    line=dict(color='#10b981', width=3),
    fill='tozeroy',
    fillcolor='rgba(16, 185, 129, 0.1)',
    hovertemplate='<b>%{x|%Y-%m-%d}</b><br>Value: $%{y:,.0f}<extra></extra>'
))

# Add initial value reference line
fig_portfolio.add_hline(y=initial_value, line_dash="dash", line_color="rgba(255,255,255,0.2)", 
                       annotation_text="Initial", annotation_position="right")

fig_portfolio.update_layout(
    title=f'<b>Portfolio Value Over Time</b><br><sub>Starting Value: ${initial_value:,.0f} | Final Value: ${final_value:,.0f} | Total Return: {total_return_pct:.2f}%</sub>',
    xaxis_title='Date',
    yaxis_title='Portfolio Value ($)',
    height=600,
    hovermode='x unified',
    plot_bgcolor=THEME_COLORS['panel'],
    paper_bgcolor=THEME_COLORS['bg'],
    font=dict(family="Arial, sans-serif", size=11, color=THEME_COLORS['text']),
    margin=dict(l=100, r=30, t=100, b=60),
    title_font=dict(size=14, color=THEME_COLORS['text'])
)

fig_portfolio.update_xaxes(
    gridcolor=THEME_COLORS['grid'],
    tickfont=dict(color=THEME_COLORS['muted']),
    showspikes=True, spikemode='across', spikethickness=1.5,
    spikecolor=THEME_COLORS['spike']
)
fig_portfolio.update_yaxes(
    gridcolor=THEME_COLORS['grid'],
    tickfont=dict(color=THEME_COLORS['muted']),
    showspikes=True, spikethickness=1.5,
    spikecolor=THEME_COLORS['spike']
)

print("✅ Displaying portfolio value chart")
fig_portfolio.show()

# Create dual chart: cumulative return % and daily returns distribution
fig_dual = make_subplots(
    rows=2, cols=1,
    subplot_titles=("Cumulative Return %", "Daily Returns Distribution"),
    vertical_spacing=0.15,
    row_heights=[0.6, 0.4]
)

# Top: Cumulative return %
cumulative_return_pct = ((portfolio_value / initial_value) - 1) * 100
fig_dual.add_trace(
    go.Scatter(
        x=portfolio_value.index,
        y=cumulative_return_pct.values,
        name='Cumulative Return %',
        mode='lines',
        line=dict(color='#3b82f6', width=2),
        fill='tozeroy',
        fillcolor='rgba(59, 130, 246, 0.1)',
        hovertemplate='<b>%{x|%Y-%m-%d}</b><br>Return: %{y:.2f}%<extra></extra>'
    ),
    row=1, col=1
)

fig_dual.add_hline(y=0, line_dash="dash", line_color="rgba(255,255,255,0.2)", row=1, col=1)

# Bottom: Daily returns distribution
fig_dual.add_trace(
    go.Histogram(
        x=portfolio_returns.values * 100,
        name='Daily Returns %',
        nbinsx=50,
        marker=dict(color='#8b5cf6'),
        hovertemplate='Return Range: %{x:.3f}%<br>Frequency: %{y}<extra></extra>'
    ),
    row=2, col=1
)

fig_dual.update_xaxes(title_text="Date", row=1, col=1)
fig_dual.update_yaxes(title_text="Cumulative Return (%)", row=1, col=1)
fig_dual.update_xaxes(title_text="Daily Return (%)", row=2, col=1)
fig_dual.update_yaxes(title_text="Frequency", row=2, col=1)

fig_dual.update_layout(
    title='<b>Portfolio Returns Analysis</b>',
    height=800,
    plot_bgcolor=THEME_COLORS['panel'],
    paper_bgcolor=THEME_COLORS['bg'],
    font=dict(family="Arial, sans-serif", size=11, color=THEME_COLORS['text']),
    margin=dict(l=70, r=30, t=80, b=60),
    title_font=dict(size=14, color=THEME_COLORS['text']),
    showlegend=False
)

fig_dual.update_xaxes(
    gridcolor=THEME_COLORS['grid'],
    tickfont=dict(color=THEME_COLORS['muted'])
)
fig_dual.update_yaxes(
    gridcolor=THEME_COLORS['grid'],
    tickfont=dict(color=THEME_COLORS['muted'])
)

print("✅ Displaying returns analysis")
fig_dual.show()

print("\n✅ Portfolio value analysis complete")


PORTFOLIO VALUE & RETURNS ANALYSIS

💰 PORTFOLIO PERFORMANCE SUMMARY (Starting with $500,000)
═════════════════════════════════════════════════════════════════════════════

FINAL RESULTS:
  Initial Portfolio Value:  $        500,000
  Final Portfolio Value:    $        508,370
  Total Return ($):         $          8,370
  Total Return (%):                    1.67%

PERFORMANCE METRICS:
  Mean Daily Return:                 0.0025%
  Daily Std Deviation:               0.3438%
  Sharpe Ratio (annual):              0.113
  Max Drawdown:                       -5.49%

  Backtest Period:          2022-07-01 to 2025-12-11
  Trading Days:                         892

═════════════════════════════════════════════════════════════════════════════


📅 MONTHLY RETURNS (Year-Month: Return $):
--------------------------------------------------------------------------------


C:\Users\andre\AppData\Local\Temp\ipykernel_35932\472527526.py:61: FutureWarning:

'M' is deprecated and will be removed in a future version, please use 'ME' instead.

C:\Users\andre\AppData\Local\Temp\ipykernel_35932\472527526.py:62: FutureWarning:

'M' is deprecated and will be removed in a future version, please use 'ME' instead.



,Date,Return $,Return %,Portfolio Value
30,2025-01,-7276.921662,-1.403932,510797.750093
31,2025-02,6030.584021,1.179114,516828.334114
32,2025-03,6191.756720,1.222530,523020.090834
33,2025-04,-11373.032524,-2.048081,511647.058310
34,2025-05,-716.335587,-0.128650,510930.722723
35,2025-06,-9099.736049,-1.791240,501830.986675
36,2025-07,-2331.489058,-0.461323,499499.497617
37,2025-08,4072.962247,0.816515,503572.459864
38,2025-09,433.546542,0.090581,504006.006407
39,2025-10,3993.443376,0.796205,507999.449783



📈 Creating portfolio value chart over time...

✅ Displaying portfolio value chart


✅ Displaying returns analysis



✅ Portfolio value analysis complete


## DSL Strategy Definition

Using the Quantitative DSL to define the same momentum strategy in a declarative, composable way


In [ ]:
# STAGE 1: Import DSL Components
print("="*80)
print("STAGE 1: IMPORT DSL COMPONENTS")
print("="*80)

from quantdsl_backtest.dsl.strategy import Strategy
from quantdsl_backtest.dsl.data_config import DataConfig
from quantdsl_backtest.dsl.universe import Universe, HasHistory, MinPrice
from quantdsl_backtest.dsl.factors import (
    ReturnFactor,
    VolatilityFactor,
    WinsorizedFactor,
)
from quantdsl_backtest.dsl.signals import (
    CrossSectionRank,
    MaskFromBoolean,
    NotNull,
)
from quantdsl_backtest.dsl.portfolio import (
    LongShortPortfolio,
    Book,
    TopN,
    BottomN,
    EqualWeight,
)
from quantdsl_backtest.dsl.execution import (
    Execution,
    OrderPolicy,
    LatencyModel,
    VolumeParticipation,
    PowerLawSlippageModel,
)
from quantdsl_backtest.dsl.costs import (
    Costs,
    Commission,
    BorrowCost,
    FinancingCost,
    StaticFees,
)
from quantdsl_backtest.dsl.backtest_config import BacktestConfig, Reporting

print("\n✅ All DSL components imported successfully")
print("\nAvailable components:")
print("  • Strategy, DataConfig, Universe")
print("  • Factors: ReturnFactor, VolatilityFactor, WinsorizedFactor")
print("  • Signals: CrossSectionRank, MaskFromBoolean, NotNull")
print("  • Portfolio: LongShortPortfolio, Book, TopN, BottomN, EqualWeight")
print("  • Execution: Execution, OrderPolicy, LatencyModel, VolumeParticipation, PowerLawSlippageModel")
print("  • Costs: Costs, Commission, BorrowCost, FinancingCost, StaticFees")
print("  • BacktestConfig, Reporting")


STAGE 1: IMPORT DSL COMPONENTS

✅ All DSL components imported successfully

Available components:
  • Strategy, DataConfig, Universe
  • Factors: ReturnFactor, VolatilityFactor, WinsorizedFactor
  • Signals: CrossSectionRank, MaskFromBoolean, NotNull
  • Portfolio: LongShortPortfolio, Book, TopN, BottomN, EqualWeight
  • Execution: Execution, OrderPolicy, LatencyModel, VolumeParticipation, PowerLawSlippageModel
  • Costs: Costs, Commission, BorrowCost, FinancingCost, StaticFees
  • BacktestConfig, Reporting


In [ ]:
# STAGE 2: Define Data Config & Universe
print("\n" + "="*80)
print("STAGE 2: DATA CONFIG & UNIVERSE DEFINITION")
print("="*80)

# Define where data comes from and its characteristics
data_config = DataConfig(
    source="parquet://equities/indicies.parquet",
    calendar="XNYS",
    frequency="1d",
    start="2022-07-01",
    end="2025-12-11",
    price_adjustment="split_dividend",
    fields=["close", "volume"],
)

print("\n📊 DATA CONFIGURATION:")
print(f"   Source:     {data_config.source}")
print(f"   Calendar:   {data_config.calendar}")
print(f"   Frequency:  {data_config.frequency}")
print(f"   Period:     {data_config.start} to {data_config.end}")
print(f"   Fields:     {data_config.fields}")

# Define the trading universe with filters
universe = Universe(
    name="Equity Indices",
    id_field="ticker",
    filters=[
        HasHistory(min_days=252),  # At least 1 year of history
        MinPrice(min_price=1.0),    # Minimum price filter
    ],
)

print("\n🌍 UNIVERSE DEFINITION:")
print(f"   Name:           {universe.name}")
print(f"   ID field:       {universe.id_field}")
print(f"   Filters applied: {len(universe.filters)}")
for i, f in enumerate(universe.filters, 1):
    print(f"      {i}. {f.__class__.__name__}")

print("\n✅ Stage 2 Complete: Data source and universe filters configured")



STAGE 2: DATA CONFIG & UNIVERSE DEFINITION

📊 DATA CONFIGURATION:
   Source:     parquet://equities/indicies.parquet
   Calendar:   XNYS
   Frequency:  1d
   Period:     2022-07-01 to 2025-12-11
   Fields:     ['close', 'volume']

🌍 UNIVERSE DEFINITION:
   Name:           Equity Indices
   ID field:       ticker
   Filters applied: 2
      1. HasHistory
      2. MinPrice

✅ Stage 2 Complete: Data source and universe filters configured


In [ ]:
# STAGE 3: Define Factors
print("\n" + "="*80)
print("STAGE 3: FACTOR DEFINITIONS")
print("="*80)

print("\n📈 Building factor computation tree...\n")

# Base factor: 6-month momentum (126 trading days)
mom_126_raw = ReturnFactor(
    name="mom_126_raw",
    field="close",
    lookback=126,
    method="log",
)
print("1️⃣ mom_126_raw")
print(f"   Type:     {mom_126_raw.__class__.__name__}")
print(f"   Lookback: {mom_126_raw.lookback} trading days (~6 months)")
print(f"   Method:   {mom_126_raw.method}")

# Winsorize to handle outliers (cap at 3 standard deviations)
mom_126 = WinsorizedFactor(
    name="mom_126",
    base=mom_126_raw,
    z=3.0,
)
print("\n2️⃣ mom_126 (DERIVED)")
print(f"   Type:        {mom_126.__class__.__name__}")
print(f"   Base factor: mom_126_raw")
print(f"   Winsorization: ±3.0 std dev (removes extreme outliers)")

# Volatility factor for risk-adjustment
vol_63 = VolatilityFactor(
    name="vol_63",
    field="close",
    lookback=63,
    method="realized",
    annualize=True,
    min_periods=10,
)
print("\n3️⃣ vol_63")
print(f"   Type:       {vol_63.__class__.__name__}")
print(f"   Lookback:   {vol_63.lookback} trading days (~3 months)")
print(f"   Annualized: {vol_63.annualize}")
print(f"   Min periods: {vol_63.min_periods}")

# Organize factors into a dictionary
factors = {
    "mom_126_raw": mom_126_raw,
    "mom_126": mom_126,
    "vol_63": vol_63,
}

print("\n" + "─"*80)
print(f"FACTOR SUMMARY: {len(factors)} factors defined")
print("─"*80)
for name, factor in factors.items():
    print(f"  ✓ {name:20s} → {factor.__class__.__name__}")

print("\n✅ Stage 3 Complete: Factor tree built")
print("\nFactor dependency graph:")
print("  mom_126_raw (raw returns)")
print("      ↓")
print("  mom_126 (winsorized)")
print("")
print("  vol_63 (volatility)")



STAGE 3: FACTOR DEFINITIONS

📈 Building factor computation tree...

1️⃣ mom_126_raw
   Type:     ReturnFactor
   Lookback: 126 trading days (~6 months)
   Method:   log

2️⃣ mom_126 (DERIVED)
   Type:        WinsorizedFactor
   Base factor: mom_126_raw
   Winsorization: ±3.0 std dev (removes extreme outliers)

3️⃣ vol_63
   Type:       VolatilityFactor
   Lookback:   63 trading days (~3 months)
   Annualized: True
   Min periods: 10

────────────────────────────────────────────────────────────────────────────────
FACTOR SUMMARY: 3 factors defined
────────────────────────────────────────────────────────────────────────────────
  ✓ mom_126_raw          → ReturnFactor
  ✓ mom_126              → WinsorizedFactor
  ✓ vol_63               → VolatilityFactor

✅ Stage 3 Complete: Factor tree built

Factor dependency graph:
  mom_126_raw (raw returns)
      ↓
  mom_126 (winsorized)

  vol_63 (volatility)


In [ ]:
# STAGE 4: Define Signals
print("\n" + "="*80)
print("STAGE 4: SIGNAL DEFINITIONS")
print("="*80)

print("\n📡 Composing trading signals from factors...\n")

# Signal 1: Cross-sectional percentile rank of momentum
rank_mom = CrossSectionRank(
    factor_name="mom_126",
    mask_name=None,
    method="percentile",
    name="rank_mom_126",
)
print("1️⃣ rank_mom_126")
print(f"   Type:        {rank_mom.__class__.__name__}")
print(f"   Factor used: mom_126 (winsorized 6-month momentum)")
print(f"   Method:      {rank_mom.method}")
print(f"   Output:      Percentile rank 0.0 (worst) to 1.0 (best) within each date")

# Signal 2: Validity mask - exclude null values
valid = MaskFromBoolean(
    name="valid",
    expr=NotNull(factor_name="mom_126"),
)
print("\n2️⃣ valid")
print(f"   Type:       {valid.__class__.__name__}")
print(f"   Condition:  NotNull(mom_126)")
print(f"   Purpose:    Filter out instruments with missing momentum values")
print(f"   Output:     Boolean mask (True if momentum is not NaN)")

# Organize signals into a dictionary
signals = {
    "rank_mom_126": rank_mom,
    "valid": valid,
}

print("\n" + "─"*80)
print(f"SIGNAL SUMMARY: {len(signals)} signals defined")
print("─"*80)
for name, signal in signals.items():
    print(f"  ✓ {name:20s} → {signal.__class__.__name__}")

print("\n✅ Stage 4 Complete: Trading signals composed")
print("\nSignal computation:")
print("  Input: Factors (mom_126, vol_63)")
print("    ↓")
print("  Processing: Cross-sectional rank + validity check")
print("    ↓")
print("  Output: rank_mom_126 (0.0-1.0), valid (boolean)")



STAGE 4: SIGNAL DEFINITIONS

📡 Composing trading signals from factors...

1️⃣ rank_mom_126
   Type:        CrossSectionRank
   Factor used: mom_126 (winsorized 6-month momentum)
   Method:      percentile
   Output:      Percentile rank 0.0 (worst) to 1.0 (best) within each date

2️⃣ valid
   Type:       MaskFromBoolean
   Condition:  NotNull(mom_126)
   Purpose:    Filter out instruments with missing momentum values
   Output:     Boolean mask (True if momentum is not NaN)

────────────────────────────────────────────────────────────────────────────────
SIGNAL SUMMARY: 2 signals defined
────────────────────────────────────────────────────────────────────────────────
  ✓ rank_mom_126         → CrossSectionRank
  ✓ valid                → MaskFromBoolean

✅ Stage 4 Complete: Trading signals composed

Signal computation:
  Input: Factors (mom_126, vol_63)
    ↓
  Processing: Cross-sectional rank + validity check
    ↓
  Output: rank_mom_126 (0.0-1.0), valid (boolean)


In [ ]:
# STAGE 5: Define Portfolio Construction
print("\n" + "="*80)
print("STAGE 5: PORTFOLIO CONSTRUCTION")
print("="*80)

print("\n💼 Defining long/short portfolio structure...\n")

# Calculate expected number of positions (top/bottom 50%)
# With 7 indices, roughly 3-4 per side
num_indices = 7
expected_long_count = int(np.ceil(num_indices / 2))
expected_short_count = int(np.floor(num_indices / 2))

# Long book: Top N by momentum rank (top 50%)
long_selector = TopN(
    factor_name="rank_mom_126",
    n=expected_long_count,
    mask_name=None,
)
long_book = Book(
    name="long",
    selector=long_selector,
    weighting=EqualWeight(),
)
print("1️⃣ LONG BOOK")
print(f"   Name:               {long_book.name}")
print(f"   Selector:           Top {expected_long_count} by rank_mom_126")
print(f"   Weighting:          Equal-weight within book")
print(f"   Expected positions: ~{expected_long_count} (from {num_indices} indices)")
print(f"   Target allocation:  ~50% of portfolio")

# Short book: Bottom N by momentum rank (bottom 50%)
short_selector = BottomN(
    factor_name="rank_mom_126",
    n=expected_short_count,
    mask_name=None,
)
short_book = Book(
    name="short",
    selector=short_selector,
    weighting=EqualWeight(),
)
print("\n2️⃣ SHORT BOOK")
print(f"   Name:               {short_book.name}")
print(f"   Selector:           Bottom {expected_short_count} by rank_mom_126")
print(f"   Weighting:          Equal-weight within book")
print(f"   Expected positions: ~{expected_short_count} (from {num_indices} indices)")
print(f"   Target allocation:  ~50% of portfolio")

# Combine into long/short portfolio
portfolio = LongShortPortfolio(
    long_book=long_book,
    short_book=short_book,
    rebalance_frequency="1d",
    rebalance_at="market_close",
    target_gross_leverage=1.0,  # 50% long + 50% short = 100% gross exposure
    target_net_exposure=0.0,    # Market neutral
)

print("\n" + "─"*80)
print(f"PORTFOLIO SUMMARY")
print("─"*80)
print(f"  Rebalance frequency:      {portfolio.rebalance_frequency}")
print(f"  Rebalance at:             {portfolio.rebalance_at}")
print(f"  Target gross leverage:    {portfolio.target_gross_leverage * 100:.0f}%")
print(f"  Target net exposure:      {portfolio.target_net_exposure * 100:.0f}%")
print(f"  Max weight per name:      {portfolio.max_abs_weight_per_name * 100:.1f}%")

print("\n✅ Stage 5 Complete: Portfolio structure configured")
print("\nPortfolio composition:")
print(f"  {'':8} Long 50%       │      Short 50%")
print(f"  ──────────────────────────────────────────")
print(f"  Daily │ {expected_long_count} positions (EW) │ {expected_short_count} positions (EW)")
print(f"        │ top-ranked     │ bottom-ranked")



STAGE 5: PORTFOLIO CONSTRUCTION

💼 Defining long/short portfolio structure...

1️⃣ LONG BOOK
   Name:               long
   Selector:           Top 4 by rank_mom_126
   Weighting:          Equal-weight within book
   Expected positions: ~4 (from 7 indices)
   Target allocation:  ~50% of portfolio

2️⃣ SHORT BOOK
   Name:               short
   Selector:           Bottom 3 by rank_mom_126
   Weighting:          Equal-weight within book
   Expected positions: ~3 (from 7 indices)
   Target allocation:  ~50% of portfolio

────────────────────────────────────────────────────────────────────────────────
PORTFOLIO SUMMARY
────────────────────────────────────────────────────────────────────────────────
  Rebalance frequency:      1d
  Rebalance at:             market_close
  Target gross leverage:    100%
  Target net exposure:      0%
  Max weight per name:      3.0%

✅ Stage 5 Complete: Portfolio structure configured

Portfolio composition:
           Long 50%       │      Short 50%
  ─────

In [ ]:
# STAGE 6: EXECUTION MODEL & COST STRUCTURE

print("=" * 80)
print("STAGE 6: EXECUTION MODEL & COST STRUCTURE")
print("=" * 80)
print()

# ⚡ EXECUTION MODEL
print("⚡ Configuring order execution...")
print()

order_policy = OrderPolicy(
    default_order_type="MOC",           # Market-on-close orders
    time_in_force="DAY"                 # Valid for trading day only
)

latency = LatencyModel(
    signal_to_order_delay_bars=0,       # Execute signal immediately (0 bar delay)
    market_latency_ms=0                 # No market latency
)

slippage = PowerLawSlippageModel(
    base_bps=1.0,                       # 1 basis point base slippage
    k=20.0,                             # Power law coefficient
    exponent=0.5                        # Power law exponent
)

volume_limits = VolumeParticipation(
    max_participation=0.10,             # Max 10% of ADV
    mode="proportional"                 # Proportional participation
)

execution = Execution(
    order_policy=order_policy,
    latency=latency,
    slippage=slippage,
    volume_limits=volume_limits
)

print("1️⃣ EXECUTION MODEL")
print(f"   Order type:         {order_policy.default_order_type}")
print(f"   Time in force:      {order_policy.time_in_force}")
print(f"   Signal-to-order:    {latency.signal_to_order_delay_bars} bars")
print(f"   Market latency:     {latency.market_latency_ms} ms")
print(f"   Slippage model:     PowerLaw (base={slippage.base_bps} bps, k={slippage.k})")
print(f"   Volume limits:      {volume_limits.max_participation*100:.0f}% ADV ({volume_limits.mode})")
print()

# 💰 COST STRUCTURE
print("💰 Configuring cost structure...")
print()

# Commission structure
commission = Commission(
    type="bps_notional",                # Basis points of notional value
    amount=1.0                          # 1 basis point per trade (0.01%)
)

# Borrow costs (for shorting)
borrow_cost = BorrowCost(
    default_annual_rate=0.0,            # Default borrow rate p.a.
    curve_name=None                     # Use default rates
)

# Financing costs (cash rate)
financing_cost = FinancingCost(
    base_rate_curve="SOFR",             # Base curve (SOFR)
    spread_bps=0.0                      # No additional spread
)

# Static fees (management and performance fees)
fees = StaticFees(
    nav_fee_annual=0.005,               # 0.5% annual management fee
    perf_fee_fraction=0.20              # 20% performance fee above hurdle
)

# Combine into Costs object
costs = Costs(
    commission=commission,
    borrow=borrow_cost,
    financing=financing_cost,
    fees=fees
)

print("2️⃣ COST STRUCTURE")
print(f"   Commission:         {commission.amount} {commission.type.replace('_', ' ')}")
print(f"   Borrow rate:        {borrow_cost.default_annual_rate*100:.2f}% p.a.")
print(f"   Financing curve:    {financing_cost.base_rate_curve} + {financing_cost.spread_bps} bps")
print(f"   Mgmt fee:           {fees.nav_fee_annual*100:.1f}% p.a.")
print(f"   Performance fee:    {fees.perf_fee_fraction*100:.0f}% of excess returns")
print()

print("✅ Stage 6 Complete: Execution and costs configured")


STAGE 6: EXECUTION MODEL & COST STRUCTURE

⚡ Configuring order execution...

1️⃣ EXECUTION MODEL
   Order type:         MOC
   Time in force:      DAY
   Signal-to-order:    0 bars
   Market latency:     0 ms
   Slippage model:     PowerLaw (base=1.0 bps, k=20.0)
   Volume limits:      10% ADV (proportional)

💰 Configuring cost structure...

2️⃣ COST STRUCTURE
   Commission:         1.0 bps notional
   Borrow rate:        0.00% p.a.
   Financing curve:    SOFR + 0.0 bps
   Mgmt fee:           0.5% p.a.
   Performance fee:    20% of excess returns

✅ Stage 6 Complete: Execution and costs configured


In [ ]:
# STAGE 7: Assemble Complete Strategy
print("\n" + "="*80)
print("STAGE 7: STRATEGY ASSEMBLY")
print("="*80)

print("\n🔧 Assembling all components into complete strategy...\n")

# Define backtest configuration
backtest_config = BacktestConfig(
    engine="event_driven",
    cash_initial=500_000.0,  # $500K initial capital
    reporting=Reporting(
        output_dir="",  # Disable file output for now
        store_trades=True,
        store_positions=True,
        metrics=["returns", "sharpe", "drawdown"],
    ),
)

print("1️⃣ Backtest Configuration:")
print(f"   Engine:       {backtest_config.engine}")
print(f"   Initial cash: ${backtest_config.cash_initial:,.0f}")
print(f"   Store trades: {backtest_config.reporting.store_trades}")
print(f"   Store pos:    {backtest_config.reporting.store_positions}")
print(f"   Metrics:      {', '.join(backtest_config.reporting.metrics)}")

print("\n2️⃣ Building Strategy object...\n")

# Build the complete strategy
strategy = Strategy(
    name="6M Momentum Cross-Sectional (Indices)",
    data=data_config,
    universe=universe,
    factors=factors,
    signals=signals,
    portfolio=portfolio,
    execution=execution,
    costs=costs,
    backtest=backtest_config,
)

print("✅ Strategy object created successfully!\n")

# Display complete strategy summary
print("═"*80)
print("COMPLETE STRATEGY SUMMARY")
print("═"*80)

print(f"\n📋 STRATEGY NAME")
print(f"   {strategy.name}\n")

print(f"📊 DATA & UNIVERSE")
print(f"   Source:        {strategy.data.source}")
print(f"   Period:        {strategy.data.start} to {strategy.data.end}")
print(f"   Frequency:     {strategy.data.frequency}")
print(f"   Universe:      {strategy.universe.name}")
print(f"   Filters:       {len(strategy.universe.filters)}\n")

print(f"📈 FACTORS ({len(strategy.factors)})")
for name in strategy.factors.keys():
    print(f"   • {name}")
print()

print(f"📡 SIGNALS ({len(strategy.signals)})")
for name in strategy.signals.keys():
    print(f"   • {name}")
print()

print(f"🎯 PORTFOLIO")
print(f"   Type:           {strategy.portfolio.__class__.__name__}")
print(f"   Rebalance freq:  {strategy.portfolio.rebalance_frequency}")
print(f"   Rebalance at:    {strategy.portfolio.rebalance_at}")
print(f"   Gross leverage:  {strategy.portfolio.target_gross_leverage}x")
print(f"   Net exposure:    {strategy.portfolio.target_net_exposure}x")
print(f"   Max weight:      {strategy.portfolio.max_abs_weight_per_name*100:.1f}%\n")

print(f"⚙️  EXECUTION")
print(f"   Order type:      {strategy.execution.order_policy.default_order_type}")
print(f"   Time in force:   {strategy.execution.order_policy.time_in_force}")
print(f"   Signal delay:    {strategy.execution.latency.signal_to_order_delay_bars} bars")
print(f"   Slippage base:   {strategy.execution.slippage.base_bps} bps")
print(f"   Max volume:      {strategy.execution.volume_limits.max_participation*100:.0f}% ADV\n")

print(f"💰 COSTS")
print(f"   Commission:      {strategy.costs.commission.amount} {strategy.costs.commission.type}")
print(f"   Borrow rate:     {strategy.costs.borrow.default_annual_rate*100:.2f}% p.a.")
print(f"   Financing:       {strategy.costs.financing.base_rate_curve}")
print(f"   Mgmt fee:        {strategy.costs.fees.nav_fee_annual*100:.1f}% p.a.")
print(f"   Perf fee:        {strategy.costs.fees.perf_fee_fraction*100:.0f}%\n")

print("="*80)
print("✅ DSL STRATEGY IMPLEMENTATION COMPLETE - ALL 7 STAGES SUCCESSFUL")
print("="*80)



STAGE 7: STRATEGY ASSEMBLY

🔧 Assembling all components into complete strategy...

1️⃣ Backtest Configuration:
   Engine:       event_driven
   Initial cash: $500,000
   Store trades: True
   Store pos:    True
   Metrics:      returns, sharpe, drawdown

2️⃣ Building Strategy object...

✅ Strategy object created successfully!

════════════════════════════════════════════════════════════════════════════════
COMPLETE STRATEGY SUMMARY
════════════════════════════════════════════════════════════════════════════════

📋 STRATEGY NAME
   6M Momentum Cross-Sectional (Indices)

📊 DATA & UNIVERSE
   Source:        parquet://equities/indicies.parquet
   Period:        2022-07-01 to 2025-12-11
   Frequency:     1d
   Universe:      Equity Indices
   Filters:       2

📈 FACTORS (3)
   • mom_126_raw
   • mom_126
   • vol_63

📡 SIGNALS (2)
   • rank_mom_126
   • valid

🎯 PORTFOLIO
   Type:           LongShortPortfolio
   Rebalance freq:  1d
   Rebalance at:    market_close
   Gross leverage:  1.0x
 

In [ ]:
# ============================================================================
# SIGNAL-TO-DATA FIT: Inspect how momentum factors map to price data
# ============================================================================
print("\n" + "="*80)
print("SIGNAL-TO-DATA FIT INSPECTION")
print("="*80)
print()

# Pick ticker to inspect (DAX was best performer)
inspect_ticker = "DAX"
print(f"📊 Inspecting: {inspect_ticker}\n")

# ============================================================================
# PART 1: RAW PRICE DATA
# ============================================================================
print("PART 1: RAW PRICE DATA FOR " + inspect_ticker)
print("-"*80)

ticker_data = df_raw_sorted[df_raw_sorted['ticker'] == inspect_ticker].copy()
ticker_data = ticker_data.sort_index()

print(f"  Records:      {len(ticker_data):,}")
print(f"  Date range:   {ticker_data.index.min().date()} to {ticker_data.index.max().date()}")
print(f"  Price range:  ${ticker_data['close'].min():.2f} - ${ticker_data['close'].max():.2f}")
print(f"  Avg volume:   {ticker_data['volume'].mean():,.0f}\n")

print("  Last 10 closing prices:")
for date, row in ticker_data[['close']].tail(10).iterrows():
    print(f"    {date.date()}: ${row['close']:.2f}")
print()

# ============================================================================
# PART 2: MOMENTUM CALCULATION
# ============================================================================
print("\nPART 2: MOMENTUM CALCULATION (126-day log returns)")
print("-"*80)

lookback = 126

# Calculate log returns
ticker_data['log_ret'] = np.log(ticker_data['close'] / ticker_data['close'].shift(1))

# Calculate momentum as sum of log returns
ticker_data['momentum_raw'] = ticker_data['log_ret'].rolling(lookback).sum()

valid_mom = ticker_data['momentum_raw'].dropna()
print(f"  Lookback period:  {lookback} days")
print(f"  Valid values:     {len(valid_mom)} out of {len(ticker_data)}")
print(f"  Mean momentum:    {valid_mom.mean():.6f}")
print(f"  Std dev:          {valid_mom.std():.6f}")
print(f"  Min/Max:          {valid_mom.min():.6f} / {valid_mom.max():.6f}\n")

print("  Last 10 momentum values:")
for date, row in ticker_data[['close', 'log_ret', 'momentum_raw']].tail(10).iterrows():
    if pd.notna(row['momentum_raw']):
        print(f"    {date.date()}: price=${row['close']:7.2f}  log_ret={row['log_ret']:8.5f}  momentum={row['momentum_raw']:9.6f}")
    else:
        print(f"    {date.date()}: (insufficient history)")
print()

# ============================================================================
# PART 3: WINSORIZATION
# ============================================================================
print("\nPART 3: WINSORIZATION (±3 standard deviations)")
print("-"*80)

mom_mean = valid_mom.mean()
mom_std = valid_mom.std()
lower_bound = mom_mean - 3*mom_std
upper_bound = mom_mean + 3*mom_std

ticker_data['momentum_winsorized'] = ticker_data['momentum_raw'].clip(lower_bound, upper_bound)

print(f"  Mean:         {mom_mean:.6f}")
print(f"  Std:          {mom_std:.6f}")
print(f"  Lower bound:  {lower_bound:.6f}  (mean - 3σ)")
print(f"  Upper bound:  {upper_bound:.6f}  (mean + 3σ)\n")

winsorized_count = (ticker_data['momentum_raw'] != ticker_data['momentum_winsorized']).sum()
print(f"  Clipped values: {winsorized_count} ({winsorized_count/len(valid_mom)*100:.2f}%)\n")

if winsorized_count > 0:
    print("  Examples of winsorized values:")
    clipped = ticker_data[ticker_data['momentum_raw'] != ticker_data['momentum_winsorized']].tail(5)
    for date, row in clipped.iterrows():
        print(f"    {date.date()}: {row['momentum_raw']:9.6f} → {row['momentum_winsorized']:9.6f}")
    print()

# ============================================================================
# PART 4: SIGNAL PERCENTILE RANKING
# ============================================================================
print("\nPART 4: SIGNAL PERCENTILE RANKING")
print("-"*80)

recent_252 = ticker_data['momentum_winsorized'].tail(252).dropna()

print(f"  Recent period: last 252 trading days (≈1 year)\n")

# Show percentile distribution
percentiles = [0, 20, 40, 50, 60, 80, 100]
print("  Momentum signal percentiles:")
for pct in percentiles:
    val = recent_252.quantile(pct/100)
    label = f"{pct:3d}th"
    if pct == 50:
        label = "MEDIAN"
    print(f"    {label:6s}: {val:10.6f}")
print()

# ============================================================================
# PART 5: LATEST SIGNAL vs RECENT HISTORY
# ============================================================================
print("\nPART 5: LATEST SIGNAL")
print("-"*80)

latest_idx = ticker_data[ticker_data['momentum_winsorized'].notna()].index[-1]
latest_mom = ticker_data.loc[latest_idx, 'momentum_winsorized']
latest_price = ticker_data.loc[latest_idx, 'close']

# Calculate percentile rank
pct_rank = (recent_252 <= latest_mom).sum() / len(recent_252) * 100

print(f"  Date:       {latest_idx.date()}")
print(f"  Price:      ${latest_price:.2f}")
print(f"  Momentum:   {latest_mom:.6f}")
print(f"  Percentile: {pct_rank:.1f}th (vs last 252 days)")
print()

if pct_rank > 80:
    print("  ⬆️  STRONG momentum signal (top 20%)")
elif pct_rank > 60:
    print("  ↗️  Good momentum signal (above median)")
elif pct_rank > 40:
    print("  → Moderate momentum signal (above 40%)")
elif pct_rank > 20:
    print("  ↘️  Weak momentum signal (below median)")
else:
    print("  ⬇️  POOR momentum signal (bottom 20%)")
print()

# ============================================================================
# SUMMARY
# ============================================================================
print("\n" + "="*80)
print("✅ SIGNAL-TO-DATA FIT INSPECTION COMPLETE")
print("="*80)
print(f"\n{inspect_ticker} Momentum Signal Summary:")
print(f"  • Raw signal from price: 126-day log returns")
print(f"  • Winsorization: clips to ±3σ to handle outliers")
print(f"  • Current status: {pct_rank:.0f}th percentile")
print(f"  • Expected trading signal: {'🟢 LONG' if pct_rank > 60 else '🔴 SHORT' if pct_rank < 40 else '⚪ NEUTRAL'}")
print()



SIGNAL-TO-DATA FIT INSPECTION

📊 Inspecting: DAX

PART 1: RAW PRICE DATA FOR DAX
--------------------------------------------------------------------------------
  Records:      7,305
  Date range:   1997-03-04 to 2025-12-12
  Price range:  $2202.96 - $24611.25
  Avg volume:   87,564,578

  Last 10 closing prices:
    2025-12-01: $23589.44
    2025-12-02: $23710.86
    2025-12-03: $23693.71
    2025-12-04: $23882.03
    2025-12-05: $24028.14
    2025-12-08: $24046.01
    2025-12-09: $24162.65
    2025-12-10: $24130.14
    2025-12-11: $24294.61
    2025-12-12: $24186.49


PART 2: MOMENTUM CALCULATION (126-day log returns)
--------------------------------------------------------------------------------
  Lookback period:  126 days
  Valid values:     7179 out of 7305
  Mean momentum:    0.032636
  Std dev:          0.154075
  Min/Max:          -0.702427 / 0.489575

  Last 10 momentum values:
    2025-12-01: price=$23589.44  log_ret=-0.01043  momentum=-0.029861
    2025-12-02: price=$237

In [ ]:
# ============================================================================
# DIRECT SIGNAL EXPLORATION - Inspect Data Structure
# ============================================================================
print("\n" + "="*80)
print("DIRECT SIGNAL EXPLORATION")
print("="*80)
print()

# First, let's understand what we have
print("📊 DATA STRUCTURE INSPECTION:")
print(f"   all_signals type: {type(all_signals)}")
print(f"   all_signals shape: {all_signals.shape}")
print(f"   Index type: {type(all_signals.index)}")
print(f"   Index nlevels: {all_signals.index.nlevels}")
print(f"   Index names: {all_signals.index.names}")
print(f"   Columns: {all_signals.columns.tolist()}")
print()

# Get list of unique dates
unique_dates = all_signals.index.get_level_values(0).unique() if all_signals.index.nlevels > 1 else all_signals.index.unique()
latest_date = unique_dates.max()

print(f"Latest date in data: {latest_date}")
print(f"Total dates: {len(unique_dates)}")
print()

# Get latest signals for this date
latest_signals_data = all_signals.loc[latest_date] if all_signals.index.nlevels > 1 else all_signals.loc[[latest_date]]

print(f"✓ Latest signals retrieved")
print(f"  Type: {type(latest_signals_data)}")
print(f"  Shape: {latest_signals_data.shape}")
print(f"  Columns: {latest_signals_data.columns.tolist()}")
print()

# Display a sample
print("📋 SAMPLE DATA (First 5 rows):")
print(latest_signals_data.head())
print()

# If we have the expected columns, show summary stats
expected_cols = ['mom_126', 'volatility', 'rank_mom_126']
available_cols = [col for col in expected_cols if col in latest_signals_data.columns]

if available_cols:
    print(f"✓ Found {len(available_cols)} signal columns: {available_cols}")
    print()
    for col in available_cols:
        print(f"  {col}:")
        print(f"    Mean:  {latest_signals_data[col].mean():+.4f}")
        print(f"    Std:   {latest_signals_data[col].std():.4f}")
        print(f"    Min:   {latest_signals_data[col].min():.4f}")
        print(f"    Max:   {latest_signals_data[col].max():.4f}")
        print()
else:
    print("⚠️  Expected signal columns not found in latest data")
    print("Available data will be used for visualizations")



DIRECT SIGNAL EXPLORATION

📊 DATA STRUCTURE INSPECTION:
   all_signals type: <class 'pandas.core.frame.DataFrame'>
   all_signals shape: (60370, 11)
   Index type: <class 'pandas.core.indexes.multi.MultiIndex'>
   Index nlevels: 2
   Index names: ['date', 'ticker']
   Columns: ['close', 'mom_1m', 'mom_3m', 'mom_6m', 'mom_9m', 'mom_12m', 'mom_1m_rank', 'mom_3m_rank', 'mom_6m_rank', 'mom_9m_rank', 'mom_12m_rank']

Latest date in data: 2025-12-12 00:00:00
Total dates: 10837

✓ Latest signals retrieved
  Type: <class 'pandas.core.frame.DataFrame'>
  Shape: (7, 11)
  Columns: ['close', 'mom_1m', 'mom_3m', 'mom_6m', 'mom_9m', 'mom_12m', 'mom_1m_rank', 'mom_3m_rank', 'mom_6m_rank', 'mom_9m_rank', 'mom_12m_rank']

📋 SAMPLE DATA (First 5 rows):
           close    mom_1m    mom_3m    mom_6m    mom_9m   mom_12m  \
ticker                                                               
CAC      8068.62 -0.019905  0.032028  0.068203 -0.012586  0.095432   
CCMP    23195.17 -0.009027  0.037873  0.195

In [ ]:
# ============================================================================
# DSL SIGNAL EXPLORER - Pandas DataFrames (DEPRECATED - Use Interactive Dashboard Above)
# ============================================================================
# ⚠️ This cell uses the old explorer API which is no longer available
# Use the Interactive Explorer Dashboard (STAGE 9) above instead
# The dashboard provides all the same analysis with a modern UI

print("="*80)
print("ℹ️  SIGNAL EXPLORATION: Use Interactive Dashboard Above")
print("="*80)
print()
print("The old pandas-based explorer has been replaced with:")
print("  • InteractiveDSLExplorer (STAGE 9)")
print("  • 4-tab dashboard: Rankings | Positions | Heatmap | Ticker Analysis")
print("  • Professional 3D visualizations (STAGE 10)")
print()
print("✅ Run STAGE 9 cells above for full interactive analysis")
print()

# ============================================================================
# Quick Signal Summary from Available Data
# ============================================================================
print("\n📊 LATEST SIGNAL SUMMARY (from all_signals)")
print("-"*80)
print()

# Get latest date data
latest_date = all_signals.index.get_level_values(0).max()
latest_signals = all_signals.loc[latest_date]

print(f"Date: {latest_date.strftime('%Y-%m-%d')}")
print()

# Momentum rankings
momentum_ranked = latest_signals.sort_values('mom_6m', ascending=False)
print("Momentum Rankings (Top 5):")
for i, (ticker, row) in enumerate(momentum_ranked.head(5).iterrows(), 1):
    print(f"  {i}. {ticker:8s} → {row['mom_6m']:7.4f}")

print()
print("Momentum Rankings (Bottom 5):")
for i, (ticker, row) in enumerate(momentum_ranked.tail(5).iterrows(), 1):
    print(f"  {i}. {ticker:8s} → {row['mom_6m']:7.4f}")

print()
print("📈 Statistics:")
print(f"  Mean momentum:     {latest_signals['mom_6m'].mean():8.4f}")
print(f"  Std dev:           {latest_signals['mom_6m'].std():8.4f}")
print(f"  Min/Max:           {latest_signals['mom_6m'].min():8.4f} / {latest_signals['mom_6m'].max():8.4f}")
print()


ℹ️  SIGNAL EXPLORATION: Use Interactive Dashboard Above

The old pandas-based explorer has been replaced with:
  • InteractiveDSLExplorer (STAGE 9)
  • 4-tab dashboard: Rankings | Positions | Heatmap | Ticker Analysis
  • Professional 3D visualizations (STAGE 10)

✅ Run STAGE 9 cells above for full interactive analysis


📊 LATEST SIGNAL SUMMARY (from all_signals)
--------------------------------------------------------------------------------

Date: 2025-12-12

Momentum Rankings (Top 5):
  1. IBEX     →  0.2262
  2. CCMP     →  0.1952
  3. SPX      →  0.1423
  4. MIB      →  0.1039
  5. UKX      →  0.0911

Momentum Rankings (Bottom 5):
  1. SPX      →  0.1423
  2. MIB      →  0.1039
  3. UKX      →  0.0911
  4. CAC      →  0.0682
  5. DAX      →  0.0490

📈 Statistics:
  Mean momentum:       0.1251
  Std dev:             0.0659
  Min/Max:             0.0490 /   0.2262



In [ ]:
# ============================================================================
# DSL SIGNAL EXPLORER - Visual Exploration (DEPRECATED)
# ============================================================================
# ⚠️ This cell uses the old explorer API which is no longer available
# Use the Interactive Dashboard (STAGE 9) above for all visualizations

print("\n" + "="*80)
print("ℹ️  SIGNAL EXPLORATION: VISUAL ANALYSIS (Use Interactive Dashboard)")
print("="*80)
print()
print("The old visualization explorer has been replaced with:")
print("  • Interactive Explorer Dashboard (STAGE 9) - 4 tabs with charts")
print("  • Professional 3D Visualizations (STAGE 10) - Advanced analysis")
print()
print("✅ See STAGE 9 and STAGE 10 for all visualization options")
print()


print("\n📊 DAX ANALYSIS - Price, Momentum, Volatility")
print("-"*80)
display(fig_dax)

print("\n📊 UKX ANALYSIS - Price, Momentum, Volatility")
print("-"*80)
display(fig_ukx)

print("\n✅ Visual exploration complete")


In [ ]:
# TEST SUITE: DSL Strategy Inspection for Lagging Indices
print("\n" + "="*80)
print("TEST SUITE: DSL STRATEGY INSPECTION")
print("="*80)
print()

# ============================================================================
# PART 1: DSL CONFIGURATION REVIEW
# ============================================================================
print("PART 1: DSL CONFIGURATION")
print("-"*80)
print()

print("📋 STRATEGY METADATA")
print(f"   Name:          {strategy.name}")
print(f"   Engine:        {strategy.backtest.engine}")
print(f"   Initial cash:  ${strategy.backtest.cash_initial:,.0f}")
print()

print("📊 DATA SOURCE")
data_cfg = strategy.data
print(f"   Source:        {data_cfg.source}")
print(f"   Period:        {data_cfg.start} to {data_cfg.end}")
print(f"   Calendar:      {data_cfg.trading_calendar}")
print(f"   Frequency:     {data_cfg.frequency}")
print(f"   Fields:        {', '.join(data_cfg.fields)}")
print()

print("🌍 UNIVERSE & FILTERS")
univ = strategy.universe
print(f"   Name:          {univ.name}")
print(f"   ID field:      {univ.id_field}")
print(f"   Filters:       {len(univ.filters)}")
for i, filt in enumerate(univ.filters, 1):
    print(f"     {i}. {filt.__class__.__name__}")
print()

print("📈 FACTORS (3)")
for name, factor in strategy.factors.items():
    ftype = factor.__class__.__name__
    if hasattr(factor, 'base_factor'):
        print(f"   • {name:20s} ← {ftype} (base: {factor.base_factor.__class__.__name__})")
    else:
        print(f"   • {name:20s} ← {ftype}")
print()

print("📡 SIGNALS (2)")
for name, signal in strategy.signals.items():
    print(f"   • {name:20s} ← {signal.__class__.__name__}")
print()

print("🎯 PORTFOLIO RULES")
port = strategy.portfolio
print(f"   Type:          {port.__class__.__name__}")
print(f"   Rebalance:     {port.rebalance_frequency} at {port.rebalance_at}")
print(f"   Long book:     {port.long_book.selector.__class__.__name__}")
print(f"   Short book:    {port.short_book.selector.__class__.__name__}")
print(f"   Sizing:        {port.long_book.sizing.__class__.__name__} (long), {port.short_book.sizing.__class__.__name__} (short)")
print(f"   Leverage:      {port.target_gross_leverage}x gross, {port.target_net_exposure}x net")
print()

print("⚡ EXECUTION")
ex = strategy.execution
print(f"   Order type:    {ex.order_policy.default_order_type}")
print(f"   Slippage:      {ex.slippage.base_bps} bps base")
print(f"   Volume limit:  {ex.volume_limits.max_participation*100:.0f}% ADV")
print()

print("💰 COSTS")
print(f"   Commission:    {strategy.costs.commission.amount} bps")
print(f"   Mgmt fee:      {strategy.costs.fees.nav_fee_annual*100:.2f}% p.a.")
print(f"   Perf fee:      {strategy.costs.fees.perf_fee_fraction*100:.0f}%")
print()


In [ ]:
# TEST SUITE PART 2: Raw Data Inspection
print("\n" + "="*80)
print("PART 2: RAW DATA INSPECTION")
print("-"*80)
print()

# Load raw data
raw_data_path = data_path / "equities" / "sp500_daily"
if raw_data_path.exists():
    try:
        import pyarrow.parquet as pq
        parquet_files = list(raw_data_path.glob("*.parquet"))
        if parquet_files:
            # Load first file to inspect structure
            df_sample = pd.read_parquet(parquet_files[0])
            print(f"📁 Loaded from: {raw_data_path}")
            print(f"   Files found: {len(parquet_files)}")
            print()
            
            print(f"Sample data shape: {df_sample.shape}")
            print(f"Columns: {list(df_sample.columns)}")
            print(f"Date range: {df_sample.index.min()} to {df_sample.index.max()}")
            print()
            
            print("First 5 rows:")
            print(df_sample.head())
            print()
            
            print("Data statistics:")
            print(df_sample[['close', 'volume']].describe())
        else:
            print(f"❌ No parquet files found in {raw_data_path}")
    except Exception as e:
        print(f"❌ Error loading data: {e}")
else:
    print(f"❌ Data path not found: {raw_data_path}")
    print(f"   Available paths:")
    if data_path.exists():
        for p in data_path.iterdir():
            print(f"     - {p.name}")


In [ ]:
# TEST SUITE PART 3: Factor & Signal Inspection
print("\n" + "="*80)
print("PART 3: FACTOR CALCULATION & SIGNAL INSPECTION")
print("-"*80)
print()

# Use the momentum data we already calculated in earlier cells
print("✓ Using pre-calculated momentum data from backtest cells")
print()

# Get latest date in backtest
latest_backtest_date = max(ls_signals.index)
print(f"Latest backtest date: {latest_backtest_date.date()}")
print()

# Show factor values for latest date
print("📊 FACTOR VALUES (Latest Date)")
print("-"*80)
latest_mom = ls_signals[ls_signals.index == latest_backtest_date]

if not latest_mom.empty:
    latest_mom_sorted = latest_mom.sort_values('rank_mom_126', ascending=False)
    print(f"\nDate: {latest_backtest_date.date()}\n")
    print(latest_mom_sorted[['ticker', 'mom_126', 'rank_mom_126', 'valid']].to_string())
else:
    print("No data for latest date")

print()
print()

# Show position assignments on latest date
print("🎯 POSITION ASSIGNMENTS (Latest Date)")
print("-"*80)

# Recalculate positions for latest date
expected_long_count = max(1, len(available_tickers) // 2)
expected_short_count = len(available_tickers) - expected_long_count

if not latest_mom.empty:
    long_threshold = latest_mom['rank_mom_126'].quantile(1 - (expected_long_count / len(latest_mom)))
    short_threshold = latest_mom['rank_mom_126'].quantile(expected_short_count / len(latest_mom))
    
    long_pos = latest_mom[latest_mom['rank_mom_126'] > long_threshold]['ticker'].tolist()
    short_pos = latest_mom[latest_mom['rank_mom_126'] < short_threshold]['ticker'].tolist()
    
    print(f"\n✓ Long positions ({len(long_pos)}):")
    for ticker in sorted(long_pos):
        rank = latest_mom[latest_mom['ticker'] == ticker]['rank_mom_126'].values[0]
        print(f"   • {ticker:6s}  rank={rank:.3f}")
    
    print(f"\n✗ Short positions ({len(short_pos)}):")
    for ticker in sorted(short_pos):
        rank = latest_mom[latest_mom['ticker'] == ticker]['rank_mom_126'].values[0]
        print(f"   • {ticker:6s}  rank={rank:.3f}")
    
    print(f"\n→ Neutral ({len(latest_mom) - len(long_pos) - len(short_pos)})")
    neutral_pos = latest_mom[
        (latest_mom['rank_mom_126'] >= short_threshold) & 
        (latest_mom['rank_mom_126'] <= long_threshold)
    ]['ticker'].tolist()
    for ticker in sorted(neutral_pos):
        rank = latest_mom[latest_mom['ticker'] == ticker]['rank_mom_126'].values[0]
        print(f"   • {ticker:6s}  rank={rank:.3f}")

print()


In [ ]:
# TEST SUITE PART 4: Factor Heatmap & Signal Changes
print("="*80)
print("PART 4: FACTOR HEATMAP & SIGNAL DYNAMICS")
print("-"*80)
print()

# Create factor heatmap for recent period
recent_dates = ls_signals.index[-20:]  # Last 20 trading days
recent_signals = ls_signals[ls_signals.index.isin(recent_dates)]

# Pivot for heatmap
factor_heatmap = recent_signals.pivot_table(
    index='ticker', 
    columns=recent_signals.index.date, 
    values='mom_126'
)

print("📊 MOMENTUM FACTOR HEATMAP (Last 20 Days)")
print("-"*80)
print(factor_heatmap.to_string())
print()

# Calculate position changes
print("\n📊 POSITION CHANGES (Day-over-Day)")
print("-"*80)

expected_long = max(1, len(available_tickers) // 2)
expected_short = len(available_tickers) - expected_long

position_changes = []
for i in range(1, len(recent_signals.index.unique())):
    prev_date = recent_signals.index.unique()[-i-1]
    curr_date = recent_signals.index.unique()[-i]
    
    prev_data = recent_signals[recent_signals.index == prev_date].copy()
    curr_data = recent_signals[recent_signals.index == curr_date].copy()
    
    if len(prev_data) == 0 or len(curr_data) == 0:
        continue
    
    prev_long_thresh = prev_data['rank_mom_126'].quantile(1 - (expected_long / len(prev_data)))
    prev_short_thresh = prev_data['rank_mom_126'].quantile(expected_short / len(prev_data))
    
    curr_long_thresh = curr_data['rank_mom_126'].quantile(1 - (expected_long / len(curr_data)))
    curr_short_thresh = curr_data['rank_mom_126'].quantile(expected_short / len(curr_data))
    
    prev_longs = set(prev_data[prev_data['rank_mom_126'] > prev_long_thresh]['ticker'])
    prev_shorts = set(prev_data[prev_data['rank_mom_126'] < prev_short_thresh]['ticker'])
    
    curr_longs = set(curr_data[curr_data['rank_mom_126'] > curr_long_thresh]['ticker'])
    curr_shorts = set(curr_data[curr_data['rank_mom_126'] < curr_short_thresh]['ticker'])
    
    entries = curr_longs - prev_longs
    exits = prev_longs - curr_longs
    short_entries = curr_shorts - prev_shorts
    short_exits = prev_shorts - curr_shorts
    
    if entries or exits or short_entries or short_exits:
        print(f"\n{curr_date.date()}")
        if entries:
            print(f"  📈 Long entries:  {', '.join(sorted(entries))}")
        if exits:
            print(f"  📉 Long exits:    {', '.join(sorted(exits))}")
        if short_entries:
            print(f"  📈 Short entries: {', '.join(sorted(short_entries))}")
        if short_exits:
            print(f"  📉 Short exits:   {', '.join(sorted(short_exits))}")

print()
print("✅ Factor & Signal Inspection Complete")


In [ ]:
# TEST 1: DSL Configuration Inspector
print("\n" + "="*80)
print("TEST 1: DSL CONFIGURATION INSPECTOR")
print("="*80 + "\n")

import importlib
import quantdsl_backtest.dsl.inspector
importlib.reload(quantdsl_backtest.dsl.inspector)

from quantdsl_backtest.dsl.inspector import DSLConfigInspector

config_inspector = DSLConfigInspector(strategy)
print(config_inspector.summary())


In [ ]:
# TEST 2: Data Inspector - Lagging Indices
print("\n" + "="*80)
print("TEST 2: DATA INSPECTOR - LAGGING INDICES")
print("="*80 + "\n")

from quantdsl_backtest.dsl.inspector import DataInspector

data_inspector = DataInspector(data_path)
print(data_inspector.data_summary())

# Show data for each ticker
print("\nPER-TICKER DATA SUMMARY:")
print("-" * 80)
for ticker in available_tickers:
    ticker_df = data_inspector.get_ticker_data(ticker)
    print(f"\n{ticker}:")
    print(f"  Records:    {len(ticker_df):,}")
    print(f"  Date range: {ticker_df.index.min().date()} to {ticker_df.index.max().date()}")
    if 'close' in ticker_df.columns:
        print(f"  Close range: ${ticker_df['close'].min():.2f} - ${ticker_df['close'].max():.2f}")
    if 'volume' in ticker_df.columns:
        print(f"  Avg volume:  {ticker_df['volume'].mean():,.0f}")


In [ ]:
# TEST 3: Signal Inspector - Momentum Rankings
print("\n" + "="*80)
print("TEST 3: SIGNAL INSPECTOR - MOMENTUM RANKINGS")
print("="*80 + "\n")

# Get the latest signals from our backtest
latest_date = all_signals.index.max()
latest_signals = all_signals.loc[latest_date].copy()
latest_signals['ticker'] = latest_signals.index

print(f"Signal Date: {latest_date.date()}")
print(f"\nTop 5 Momentum Leaders:")
print("-" * 60)
top5 = latest_signals.nlargest(5, 'rank_mom_126')[['ticker', 'mom_126', 'rank_mom_126']]
for idx, (_, row) in enumerate(top5.iterrows(), 1):
    print(f"  {idx}. {row['ticker']:6s} | Momentum: {row['mom_126']:7.2%} | Rank: {row['rank_mom_126']:5.1%}")

print(f"\nBottom 5 Momentum Laggards:")
print("-" * 60)
bottom5 = latest_signals.nsmallest(5, 'rank_mom_126')[['ticker', 'mom_126', 'rank_mom_126']]
for idx, (_, row) in enumerate(bottom5.iterrows(), 1):
    print(f"  {idx}. {row['ticker']:6s} | Momentum: {row['mom_126']:7.2%} | Rank: {row['rank_mom_126']:5.1%}")

# Signal distribution
print(f"\nSignal Distribution (All tickers):")
print("-" * 60)
print(f"  Mean rank: {latest_signals['rank_mom_126'].mean():5.1%}")
print(f"  Median:    {latest_signals['rank_mom_126'].median():5.1%}")
print(f"  Std dev:   {latest_signals['rank_mom_126'].std():5.1%}")
print(f"  Min/Max:   {latest_signals['rank_mom_126'].min():5.1%} / {latest_signals['rank_mom_126'].max():5.1%}")


In [ ]:
# STAGE 9: INTERACTIVE EXPLORER DASHBOARD

print("=" * 80)
print("STAGE 9: INTERACTIVE EXPLORER DASHBOARD")
print("=" * 80)
print()

from quantdsl_backtest.dsl.explorer import InteractiveDSLExplorer

# Create interactive explorer
dsl_explorer = InteractiveDSLExplorer(
    strategy=strategy,
    data_path=data_path,
    tickers=available_tickers
)

print("🎛️ Creating interactive dashboard...")
print()

# Display the explorer with 4 tabs
explorer_ui = dsl_explorer.create_explorer()
display(explorer_ui)

## Stage 10: 3D Signal Visualization

Advanced 3D visualizations for signal-to-data exploration using professional charting libraries.

In [ ]:
# Install professional 3D charting libraries
import subprocess
import sys

packages = ['plotly', 'plotly-orca', 'kaleido', 'vispy']
print("Installing 3D visualization libraries...")

for package in packages:
    try:
        __import__(package)
        print(f"✅ {package} already installed")
    except ImportError:
        print(f"📦 Installing {package}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", package, "-q"])
        print(f"✅ {package} installed")

print("\n✅ All visualization libraries ready!")


## 🎯 PROFESSIONAL SIGNAL ANALYSIS DASHBOARD

**Time-centric visualization suite with TradingView-inspired styling**

The following charts link together to show statistical signals evolving over time:
1. **3D Momentum Timeline** - Time × Indices × Momentum (main chart)
2. **Momentum Heatmap** - Professional heatmap of momentum evolution  
3. **Regime Transitions** - When signals enter/exit bull/bear/neutral regimes
4. **Forward Returns** - Predictive power by signal strength (separate axis)
5. **Price Correlation** - How momentum predicts next 1/5/20 day returns (separate chart)

In [ ]:
# === 2D COMPANION CHARTS: Asset Returns & Momentum (Stacked with Shared Timeline) ===
from plotly.subplots import make_subplots

# Prepare 2D data - last 60 days
recent_dates_2d = all_signals.index.get_level_values(0).unique()[-60:]
colors_ticker = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#FFA07A', '#98D8C8', '#F7DC6F', '#BB8FCE']

# Create subplots: 2 rows, 1 column - shared x-axis
fig = make_subplots(
    rows=2, cols=1,
    shared_xaxes=True,
    vertical_spacing=0.12,
    subplot_titles=('Asset Returns Over Time', 'Momentum Signal Evolution'),
    specs=[[{"secondary_y": False}], [{"secondary_y": False}]]
)

# Collect data for both charts
data_by_ticker = {}

for ticker_idx, ticker in enumerate(tickers):
    returns_list = []
    momentum_list = []
    dates_list = []
    
    for date in recent_dates_2d:
        try:
            row = all_signals.loc[(date, ticker)]
            returns_list.append(row['close'])
            momentum_list.append(row['mom_6m'] * 100)
            dates_list.append(date)
        except:
            pass
    
    if len(returns_list) > 1:
        # Normalize returns to percentage
        normalized_returns = [(p / returns_list[0] - 1) * 100 for p in returns_list]
        data_by_ticker[ticker] = {
            'dates': dates_list,
            'returns': normalized_returns,
            'momentum': momentum_list,
            'color': colors_ticker[ticker_idx % len(colors_ticker)]
        }

# Add traces to subplots
for ticker, data in data_by_ticker.items():
    # Row 1: Asset Returns (TOP)
    fig.add_trace(
        go.Scatter(
            x=data['dates'],
            y=data['returns'],
            mode='lines+markers',
            name=ticker,
            line=dict(color=data['color'], width=2),
            marker=dict(size=4),
            hovertemplate=f'<b>{ticker}</b><br>Date: %{{x}}<br>Return: %{{y:.2f}}%<extra></extra>',
            legendgroup=ticker,
            showlegend=True
        ),
        row=1, col=1
    )
    
    # Row 2: Momentum (SECONDARY - below)
    fig.add_trace(
        go.Scatter(
            x=data['dates'],
            y=data['momentum'],
            mode='lines+markers',
            name=ticker,
            line=dict(color=data['color'], width=2),
            marker=dict(size=4),
            hovertemplate=f'<b>{ticker}</b><br>Date: %{{x}}<br>6M Momentum: %{{y:.2f}}%<extra></extra>',
            legendgroup=ticker,
            showlegend=False
        ),
        row=2, col=1
    )

# Add zero line to momentum chart
fig.add_hline(y=0, line_dash="dash", line_color="gray", opacity=0.5, row=2, col=1)

# Update layout - unified styling
fig.update_layout(
    title='<b>Signal Dashboard: Asset Returns & Momentum (Linked Timeline)</b><br><sub>Hover vertically to see synchronized data across both charts</sub>',
    template='plotly_dark',
    hovermode='x unified',
    width=1200,
    height=800,
    plot_bgcolor='rgba(17, 17, 17, 0.9)',
    paper_bgcolor='rgba(17, 17, 17, 0.9)',
    font=dict(color='#999', size=11),
    showlegend=True,
    legend=dict(
        orientation="v",
        yanchor="top",
        y=0.99,
        xanchor="right",
        x=0.99,
        bgcolor="rgba(0,0,0,0.5)",
        bordercolor="rgb(100,100,100)",
        borderwidth=1
    )
)

# Update y-axes
fig.update_yaxes(title_text="Cumulative Return (%)", gridcolor='rgb(50, 50, 50)', row=1, col=1)
fig.update_yaxes(title_text="6M Momentum (%)", gridcolor='rgb(50, 50, 50)', row=2, col=1)
fig.update_xaxes(title_text="Date", gridcolor='rgb(50, 50, 50)', row=2, col=1)

fig.show()
print(f"✅ Unified 2D Dashboard: {len(tickers)} indices over {len(recent_dates_2d)} days (linked timeline with vertical hover)")


✅ Unified 2D Dashboard: 7 indices over 60 days (linked timeline with vertical hover)


In [ ]:
# === LIGHTWEIGHT CHARTS VERSION (TradingView Style) ===
# Using chart.js as alternative (lighter, more reliable than LightweightCharts CDN)

from IPython.display import HTML
import json

# Prepare data for charts
recent_dates_lw = all_signals.index.get_level_values(0).unique()[-60:]
colors_ticker_lw = {
    tickers[0]: 'rgb(255, 107, 107)',    # Red
    tickers[1]: 'rgb(78, 205, 196)',     # Teal
    tickers[2]: 'rgb(69, 183, 209)',     # Blue
    tickers[3]: 'rgb(255, 160, 122)',    # Light Salmon
    tickers[4]: 'rgb(152, 216, 200)',    # Mint
    tickers[5]: 'rgb(247, 220, 111)',    # Yellow
    tickers[6]: 'rgb(187, 143, 206)',    # Purple
}

# Build chart data
chart_returns_labels = []
chart_momentum_labels = []
chart_data_returns = {ticker: [] for ticker in tickers}
chart_data_momentum = {ticker: [] for ticker in tickers}
first_returns = None

for date_idx, date in enumerate(recent_dates_lw):
    chart_returns_labels.append(date.strftime('%Y-%m-%d'))
    chart_momentum_labels.append(date.strftime('%m-%d'))
    
    date_return_values = {}
    date_momentum_values = {}
    
    for ticker in tickers:
        try:
            row = all_signals.loc[(date, ticker)]
            date_return_values[ticker] = row['close']
            date_momentum_values[ticker] = row['mom_6m'] * 100
        except:
            pass
    
    # Initialize first values for return calculation
    if date_idx == 0:
        first_returns = {t: v for t, v in date_return_values.items()}
    
    for ticker in tickers:
        if ticker in first_returns and ticker in date_return_values:
            ret_pct = (date_return_values[ticker] / first_returns[ticker] - 1) * 100
            chart_data_returns[ticker].append(round(ret_pct, 2))
            chart_data_momentum[ticker].append(round(date_momentum_values[ticker], 2))

# Create datasets for Chart.js
datasets_returns = []
datasets_momentum = []

for ticker in tickers:
    datasets_returns.append({
        'label': ticker,
        'data': chart_data_returns[ticker],
        'borderColor': colors_ticker_lw[ticker],
        'backgroundColor': colors_ticker_lw[ticker].replace('rgb', 'rgba').replace(')', ', 0.1)'),
        'borderWidth': 2,
        'fill': False,
        'tension': 0.3,
        'pointRadius': 3,
        'pointBackgroundColor': colors_ticker_lw[ticker],
    })
    
    datasets_momentum.append({
        'label': ticker,
        'data': chart_data_momentum[ticker],
        'borderColor': colors_ticker_lw[ticker],
        'backgroundColor': colors_ticker_lw[ticker].replace('rgb', 'rgba').replace(')', ', 0.1)'),
        'borderWidth': 2,
        'fill': False,
        'tension': 0.3,
        'pointRadius': 3,
        'pointBackgroundColor': colors_ticker_lw[ticker],
    })

html_template = f"""
<style>
    .chart-container {{
        position: relative;
        font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
        background: #0a0a0a;
        padding: 20px;
        border-radius: 8px;
        color: #999;
    }}
    .chart-section {{
        margin: 20px 0;
    }}
    .chart-title {{
        padding: 10px;
        color: #fff;
        font-size: 14px;
        font-weight: 600;
        background: rgba(255,255,255,0.05);
        border-left: 3px solid #1f77b4;
        border-radius: 4px;
        margin-bottom: 10px;
    }}
    .chart-subtitle {{
        font-size: 11px;
        color: #666;
        margin: 5px 10px 10px 10px;
    }}
    .chart-wrapper {{
        position: relative;
        height: 380px;
        width: 100%;
        border: 1px solid #222;
        border-radius: 4px;
        background: #111;
        padding: 10px;
        box-sizing: border-box;
    }}
</style>

<div class="chart-container">
    <div class="chart-section">
        <div class="chart-title">📊 Asset Returns (Interactive Chart)</div>
        <div class="chart-subtitle">Cumulative return (%) - 60 day view with hover details</div>
        <div class="chart-wrapper">
            <canvas id="returnsChart"></canvas>
        </div>
    </div>
    
    <div class="chart-section">
        <div class="chart-title">📈 Momentum Signal (Linked Timeline)</div>
        <div class="chart-subtitle">6-month momentum (%) - Synchronized timeline with asset returns</div>
        <div class="chart-wrapper">
            <canvas id="momentumChart"></canvas>
        </div>
    </div>
</div>

<script src="https://cdn.jsdelivr.net/npm/chart.js@3.9.1/dist/chart.min.js"></script>
<script>
(async function() {{
    // Wait for Chart.js to load
    if (!window.Chart) {{
        setTimeout(() => location.reload(), 1000);
        return;
    }}
    
    const chartConfig = {{
        responsive: true,
        maintainAspectRatio: false,
        plugins: {{
            legend: {{
                labels: {{ color: '#999', font: {{ size: 11 }} }},
            }},
        }},
        scales: {{
            x: {{
                grid: {{ color: '#222' }},
                ticks: {{ color: '#666', maxRotation: 45, minRotation: 0 }},
            }},
            y: {{
                grid: {{ color: '#222' }},
                ticks: {{ color: '#666' }},
            }},
        }},
    }};
    
    // Returns Chart
    const returnsCtx = document.getElementById('returnsChart').getContext('2d');
    const returnsChart = new Chart(returnsCtx, {{
        type: 'line',
        data: {{
            labels: {json.dumps(chart_returns_labels)},
            datasets: {json.dumps(datasets_returns)},
        }},
        options: chartConfig,
    }});
    
    // Momentum Chart
    const momentumCtx = document.getElementById('momentumChart').getContext('2d');
    const momentumChart = new Chart(momentumCtx, {{
        type: 'line',
        data: {{
            labels: {json.dumps(chart_momentum_labels)},
            datasets: {json.dumps(datasets_momentum)},
        }},
        options: chartConfig,
    }});
}})();
</script>
"""

display(HTML(html_template))
print(f"✅ Professional Charts rendered: {len(tickers)} indices over {len(recent_dates_lw)} days")
print(f"   Technology: Chart.js (more reliable than LightweightCharts CDN)")


✅ Professional Charts rendered: 7 indices over 60 days
   Technology: Chart.js (more reliable than LightweightCharts CDN)


In [ ]:
# 3D Chart 1: Time × Momentum × Rank - Scatter cloud
import plotly.graph_objects as go

# Get last 60 days for clean 3D viz
recent_dates_3d = all_signals.index.get_level_values(0).unique()[-60:]

# Collect 3D points: time index, momentum, rank
x_points, y_points, z_points, colors_pts, hover_text = [], [], [], [], []

for date_idx, date in enumerate(recent_dates_3d):
    for ticker in tickers:
        try:
            row = all_signals.loc[(date, ticker)]
            x_points.append(date_idx)
            y_points.append(row['mom_6m'])  # Momentum on Y
            z_points.append(row['mom_6m_rank'])  # Rank on Z
            colors_pts.append(row['mom_6m'])
            hover_text.append(f"{ticker}<br>Momentum: {row['mom_6m']:.2%}<br>Rank: {row['mom_6m_rank']:.2%}")
        except:
            pass

fig = go.Figure(data=[go.Scatter3d(
    x=x_points,
    y=y_points,
    z=z_points,
    mode='markers',
    marker=dict(
        size=4,
        color=colors_pts,
        colorscale='RdYlGn',
        showscale=True,
        colorbar=dict(title='Momentum', thickness=10),
        opacity=0.8,
        line=dict(width=0)
    ),
    text=hover_text,
    hovertemplate='%{text}<extra></extra>'
)])

fig.update_layout(
    title='<b>3D Signal Cloud: Time × Momentum × Rank</b>',
    scene=dict(
        xaxis_title='Time (Days)',
        yaxis_title='6M Momentum',
        zaxis_title='Momentum Rank',
        bgcolor='rgba(17, 17, 17, 0.9)',
        xaxis=dict(backgroundcolor='rgb(50, 50, 50)', gridcolor='rgb(100, 100, 100)'),
        yaxis=dict(backgroundcolor='rgb(50, 50, 50)', gridcolor='rgb(100, 100, 100)'),
        zaxis=dict(backgroundcolor='rgb(50, 50, 50)', gridcolor='rgb(100, 100, 100)')
    ),
    width=1000,
    height=700,
    template='plotly_dark'
)

fig.show()
print(f"✅ 3D Scatter plotted: {len(x_points)} data points over {len(recent_dates_3d)} days")

✅ 3D Scatter plotted: 418 data points over 60 days


### 3D Plot 2: Time Series Trajectory - Signal Evolution

Track how individual instruments move through signal space over time (3D trajectory visualization).

In [ ]:
# 3D Chart 2: Trajectories - 3D line paths for each index
recent_dates_3d = all_signals.index.get_level_values(0).unique()[-60:]
colors_list = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#FFA07A', '#98D8C8', '#F7DC6F', '#BB8FCE']

fig = go.Figure()

for ticker_idx, ticker in enumerate(tickers):
    x_traj, y_traj, z_traj = [], [], []
    
    for date_idx, date in enumerate(recent_dates_3d):
        try:
            row = all_signals.loc[(date, ticker)]
            x_traj.append(date_idx)
            y_traj.append(row['mom_6m'])
            z_traj.append(row['mom_6m_rank'])
        except:
            pass
    
    if len(x_traj) > 1:
        fig.add_trace(go.Scatter3d(
            x=x_traj,
            y=y_traj,
            z=z_traj,
            mode='lines+markers',
            name=ticker,
            line=dict(color=colors_list[ticker_idx % len(colors_list)], width=3),
            marker=dict(size=3, opacity=0.7),
            hovertemplate=f'<b>{ticker}</b><br>Momentum: %{{y:.2%}}<br>Rank: %{{z:.2%}}<extra></extra>'
        ))

fig.update_layout(
    title='<b>3D Index Trajectories: 60-Day Signal Paths</b>',
    scene=dict(
        xaxis_title='Time (Days)',
        yaxis_title='6M Momentum',
        zaxis_title='Momentum Rank',
        bgcolor='rgba(17, 17, 17, 0.9)',
        xaxis=dict(backgroundcolor='rgb(50, 50, 50)', gridcolor='rgb(100, 100, 100)'),
        yaxis=dict(backgroundcolor='rgb(50, 50, 50)', gridcolor='rgb(100, 100, 100)'),
        zaxis=dict(backgroundcolor='rgb(50, 50, 50)', gridcolor='rgb(100, 100, 100)'),
        camera=dict(eye=dict(x=1.5, y=1.5, z=1.3))
    ),
    width=1000,
    height=700,
    template='plotly_dark',
    hovermode='closest'
)

fig.show()
print(f"✅ 3D Trajectories plotted for {len(tickers)} indices")

✅ 3D Trajectories plotted for 7 indices


### 3D Plot 3: Distribution Surface - Heatmap Grid

3D surface plot showing density distribution across signal dimensions (momentum × volatility grid with height = frequency).

In [ ]:
# 3D Chart 3: Surface - Signal density over time
recent_dates_3d = all_signals.index.get_level_values(0).unique()[-60:]

# Build 3D matrix: time x momentum bins with height = count
time_steps = len(recent_dates_3d)
mom_bins = 20

# Create grid
z_surface = np.zeros((mom_bins, time_steps))
mom_edges = np.linspace(all_signals['mom_6m'].min(), all_signals['mom_6m'].max(), mom_bins+1)

for t_idx, date in enumerate(recent_dates_3d):
    for ticker in tickers:
        try:
            mom = all_signals.loc[(date, ticker), 'mom_6m']
            bin_idx = int((mom - mom_edges[0]) / (mom_edges[-1] - mom_edges[0]) * mom_bins)
            bin_idx = min(max(bin_idx, 0), mom_bins - 1)
            z_surface[bin_idx, t_idx] += 1
        except:
            pass

x_grid = np.arange(time_steps)
y_grid = (mom_edges[:-1] + mom_edges[1:]) / 2

fig = go.Figure(data=[go.Surface(
    x=x_grid,
    y=y_grid,
    z=z_surface,
    colorscale='Viridis',
    showscale=True,
    colorbar=dict(title='Count', thickness=15)
)])

fig.update_layout(
    title='<b>3D Surface: Signal Density Over Time</b>',
    scene=dict(
        xaxis_title='Time (Days)',
        yaxis_title='6M Momentum',
        zaxis_title='Signal Count',
        bgcolor='rgba(17, 17, 17, 0.9)',
        xaxis=dict(backgroundcolor='rgb(50, 50, 50)'),
        yaxis=dict(backgroundcolor='rgb(50, 50, 50)'),
        zaxis=dict(backgroundcolor='rgb(50, 50, 50)'),
        camera=dict(eye=dict(x=1.2, y=1.2, z=1.2))
    ),
    width=1000,
    height=700,
    template='plotly_dark'
)

fig.show()
print(f"✅ 3D Surface plotted: {mom_bins} momentum bins × {time_steps} days")

✅ 3D Surface plotted: 20 momentum bins × 60 days


### 3D Plot 4: Interactive Bubble Chart - Return Attribution

3D bubble chart with size = portfolio weight, color = sector, position = signal space dimensions.

In [ ]:
# 3D Chart 4: Bubble chart - Size by forward return, colored by regime
recent_dates_3d = all_signals.index.get_level_values(0).unique()[-30:]  # Last 30 days

x_bubble, y_bubble, z_bubble, sizes, colors_bubble, text_bubble = [], [], [], [], [], []

for date_idx in range(len(recent_dates_3d) - 1):
    curr_date = recent_dates_3d[date_idx]
    next_date = recent_dates_3d[date_idx + 1]
    
    for ticker in tickers:
        try:
            curr_row = all_signals.loc[(curr_date, ticker)]
            next_row = all_signals.loc[(next_date, ticker)]
            
            fwd_ret = (next_row['close'] - curr_row['close']) / curr_row['close']
            
            x_bubble.append(date_idx)
            y_bubble.append(curr_row['mom_6m'])
            z_bubble.append(curr_row['mom_6m_rank'])
            sizes.append(abs(fwd_ret) * 100 + 3)
            
            # Color by return direction
            colors_bubble.append('#2ECC71' if fwd_ret > 0 else '#E74C3C')
            text_bubble.append(f"{ticker}<br>Mom: {curr_row['mom_6m']:.2%}<br>Rank: {curr_row['mom_6m_rank']:.2%}<br>1D Ret: {fwd_ret:+.2%}")
        except:
            pass

fig = go.Figure(data=[go.Scatter3d(
    x=x_bubble,
    y=y_bubble,
    z=z_bubble,
    mode='markers',
    marker=dict(
        size=sizes,
        color=colors_bubble,
        opacity=0.7,
        line=dict(width=1, color='white')
    ),
    text=text_bubble,
    hovertemplate='%{text}<extra></extra>'
)])

fig.update_layout(
    title='<b>3D Bubble Chart: Momentum × Rank × Forward Return</b><br><sub>Green=Positive Return, Red=Negative</sub>',
    scene=dict(
        xaxis_title='Time (Days)',
        yaxis_title='6M Momentum',
        zaxis_title='Momentum Rank',
        bgcolor='rgba(17, 17, 17, 0.9)',
        xaxis=dict(backgroundcolor='rgb(50, 50, 50)'),
        yaxis=dict(backgroundcolor='rgb(50, 50, 50)'),
        zaxis=dict(backgroundcolor='rgb(50, 50, 50)')
    ),
    width=1000,
    height=700,
    template='plotly_dark',
    showlegend=False
)

fig.show()
print(f"✅ 3D Bubble chart plotted: {len(x_bubble)} signals")

✅ 3D Bubble chart plotted: 199 signals


### 3D Plot 5: Advanced - Heatmap with Sunburst (Interactive Hierarchy)

Hierarchical view of signal clusters and sub-groups with interactive drill-down capabilities.

In [ ]:
# 3D Chart 5: Quintile performance - Price × Momentum × Rank colored by quintile
recent_dates_3d = all_signals.index.get_level_values(0).unique()[-30:]

x_quint, y_quint, z_quint, colors_quint, text_quint = [], [], [], [], []
quintile_colors = {0: '#C41E3A', 1: '#E67E22', 2: '#F0E68C', 3: '#85C1E2', 4: '#2ECC71'}

for date_idx in range(len(recent_dates_3d) - 1):
    curr_date = recent_dates_3d[date_idx]
    next_date = recent_dates_3d[date_idx + 1]
    
    try:
        curr_df = all_signals.loc[curr_date].copy()
        next_df = all_signals.loc[next_date].copy()
        curr_df['fwd_ret'] = (next_df['close'] - curr_df['close']) / curr_df['close']
        curr_df['quintile'] = pd.qcut(curr_df['mom_6m'], q=5, labels=False, duplicates='drop')
        
        for ticker in tickers:
            try:
                row = curr_df.loc[ticker]
                x_quint.append(date_idx)
                y_quint.append(row['mom_6m'])
                z_quint.append(row['close'])
                q = int(row['quintile']) if pd.notna(row['quintile']) else 2
                colors_quint.append(quintile_colors.get(q, '#999'))
                text_quint.append(f"{ticker}<br>Quintile: {q+1}/5<br>Momentum: {row['mom_6m']:.2%}<br>Price: ${row['close']:.2f}<br>1D Return: {row['fwd_ret']:+.2%}")
            except:
                pass
    except:
        pass

fig = go.Figure(data=[go.Scatter3d(
    x=x_quint,
    y=y_quint,
    z=z_quint,
    mode='markers',
    marker=dict(
        size=5,
        color=colors_quint,
        opacity=0.8,
        line=dict(width=0.5, color='white')
    ),
    text=text_quint,
    hovertemplate='%{text}<extra></extra>'
)])

fig.update_layout(
    title='<b>3D Quintile Space: Time × Momentum × Price</b><br><sub>Colors: Q1 (Red) to Q5 (Green)</sub>',
    scene=dict(
        xaxis_title='Time (Days)',
        yaxis_title='6M Momentum',
        zaxis_title='Close Price ($)',
        bgcolor='rgba(17, 17, 17, 0.9)',
        xaxis=dict(backgroundcolor='rgb(50, 50, 50)'),
        yaxis=dict(backgroundcolor='rgb(50, 50, 50)'),
        zaxis=dict(backgroundcolor='rgb(50, 50, 50)'),
        camera=dict(eye=dict(x=1.3, y=1.3, z=1.2))
    ),
    width=1000,
    height=700,
    template='plotly_dark',
    showlegend=False
)

fig.show()
print(f"✅ 3D Quintile chart plotted: {len(x_quint)} signals")

✅ 3D Quintile chart plotted: 201 signals
